# Self vs. Therapist: Multi-Agent Personality-Disorder Simulation

A multi-agent "Self" (five OCEAN trait agents + consensus) is interviewed by a "Therapist" agent across DSM-5-TR personality-disorder criteria. Each session is scored against a held-out ground-truth diagnosis, validated against a rater-baseline on textbook vignettes, and averaged across independent rounds. See `PROJECT_DESCRIPTION.md` for the full write-up.

## 1. Setup: OpenRouter client, model config, `chat()` helper

In [1]:
import json
import os
import random
import re
import time
from typing import List, Literal, TypedDict

from dotenv import load_dotenv
from langgraph.graph import END, StateGraph
from openai import APIStatusError, OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# ---- Model config ----
# Set USE_PAID_MODELS=0
# in .env to return to the free baseline.

FREE_MODEL = "google/gemma-3-12b-it:free"

SELF_PAID_MODEL = "openai/gpt-oss-120b"
CONSENSUS_PAID_MODEL = "openai/gpt-oss-120b"
THERAPIST_PAID_MODEL = "google/gemini-3.7-flash"
REPORT_PAID_MODEL = "openai/gpt-oss-120b"
JUDGE_PAID_MODEL = "google/gemini-3.7-flash"

USE_PAID_MODELS = os.getenv("USE_PAID_MODELS", "1").strip().lower() in {"1", "true", "yes"}
USE_FREE_MODELS = not USE_PAID_MODELS
SELF_MODEL = SELF_PAID_MODEL if USE_PAID_MODELS else FREE_MODEL
THERAPIST_MODEL = THERAPIST_PAID_MODEL if USE_PAID_MODELS else FREE_MODEL
REPORT_MODEL = REPORT_PAID_MODEL if USE_PAID_MODELS else FREE_MODEL
EVAL_JUDGE_MODEL = JUDGE_PAID_MODEL if USE_PAID_MODELS else FREE_MODEL

# The paid setting avoids truncating structured reports; free mode stays credit-safe.
MAX_COMPLETION_TOKENS = 250 if USE_FREE_MODELS else 4000

# The OCEAN ensemble needs seven calls for every answer. A single direct Self response
# preserves case evidence better and keeps an eight-turn free run below the 50-request/day cap.
USE_TRAIT_ENSEMBLE = USE_PAID_MODELS
print(f"Model mode: {'paid' if USE_PAID_MODELS else 'free'} | Self: {SELF_MODEL} | Consensus: {CONSENSUS_PAID_MODEL} | Judge: {EVAL_JUDGE_MODEL}"
      f"Therapist: {THERAPIST_MODEL} | Report: {REPORT_MODEL}")


def _message_text(message) -> str:
    content = getattr(message, "content", None)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get("text") or item.get("content") or "")
            else:
                parts.append(getattr(item, "text", "") or getattr(item, "content", ""))
        return "".join(parts)

    extras = getattr(message, "model_extra", None) or {}
    for key in ("reasoning", "reasoning_content", "text", "refusal"):
        value = getattr(message, key, None) or extras.get(key)
        if isinstance(value, str) and value.strip():
            return value
    return ""


# Status codes worth retrying: 429 (rate-limited, often only upstream at the
# specific provider OpenRouter routed to) and the common transient 5xx errors.
# 402 (no credit) is NOT included — retrying that just wastes time and calls.
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}


def _seconds_until_retry(exc: APIStatusError, attempt: int) -> float:
    """Prefer the provider's own Retry-After header; otherwise back off
    exponentially (2, 4, 8, 16... capped at 60s) with a little jitter so many
    parallel calls don't all retry at the exact same instant."""
    retry_after = None
    response = getattr(exc, "response", None)
    header_value = getattr(response, "headers", {}).get("retry-after") if response is not None else None
    if header_value is not None:
        try:
            retry_after = float(header_value)
        except (TypeError, ValueError):
            retry_after = None
    base_wait = retry_after if retry_after is not None else min(60.0, 2.0 ** attempt)
    return base_wait + random.uniform(0, 1.0)


def chat(model, messages, temperature=0.7, reasoning_effort="low", max_tokens=MAX_COMPLETION_TOKENS,
         retries=1, max_rate_limit_retries=6):
    max_tokens = min(max_tokens, MAX_COMPLETION_TOKENS)
    last_finish_reason = None
    for attempt in range(retries + 1):
        rate_limit_attempt = 0
        while True:
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                    extra_body={"reasoning": {"effort": reasoning_effort}},
                )
                break
            except APIStatusError as exc:
                if exc.status_code == 402:
                    raise RuntimeError(
                        "OpenRouter rejected this request with HTTP 402. Free models cannot run if "
                        "your account balance is negative or this API key has no remaining credit limit. "
                        "Check OpenRouter Settings → Credits and the key's spending limit; add enough "
                        "credit to make the balance positive, then keep USE_PAID_MODELS=0 for free inference."
                    ) from exc
                if exc.status_code in RETRYABLE_STATUS_CODES and rate_limit_attempt < max_rate_limit_retries:
                    wait_s = _seconds_until_retry(exc, rate_limit_attempt)
                    print(f"  ({model} returned HTTP {exc.status_code}; retrying in {wait_s:.1f}s "
                          f"[{rate_limit_attempt + 1}/{max_rate_limit_retries}])")
                    time.sleep(wait_s)
                    rate_limit_attempt += 1
                    continue
                if exc.status_code in RETRYABLE_STATUS_CODES:
                    raise RuntimeError(
                        f"{model} kept returning HTTP {exc.status_code} after {max_rate_limit_retries} "
                        "retries with backoff. The upstream provider is still rate-limited or overloaded. "
                        "Either wait a few minutes and rerun, add your own provider key on OpenRouter "
                        "(Settings → Integrations) to get a dedicated rate limit, or switch this role to "
                        "a different model."
                    ) from exc
                raise
        
        # Some overloaded free-model providers return a syntactically successful
        # response with choices=None. Treated as a retryable empty response instead
        # of crashing with TypeError and hiding the actual provider problem.
        choices = getattr(response, "choices", None)
        if not choices:
            details = getattr(response, "model_extra", None) or {}
            last_finish_reason = f"no choices returned ({details or 'provider unavailable'})"
            continue
        choice = choices[0]
        last_finish_reason = getattr(choice, "finish_reason", None)
        text = _message_text(choice.message).strip()
        if text:
            return text

        messages = messages + [
            {
                "role": "user",
                "content": "Your previous response was empty. Please return the requested answer now, with no hidden reasoning.",
            }
        ]

    raise RuntimeError(
        f"{model} returned no usable completion after {retries + 1} attempt(s); "
        f"last status={last_finish_reason}. Free-model providers can be temporarily unavailable; "
        "wait briefly and rerun, or set USE_PAID_MODELS=1 after adding credits."
    )

Model mode: paid | Self: openai/gpt-oss-120b | Consensus: openai/gpt-oss-120b | Judge: google/gemini-3.7-flashTherapist: google/gemini-3.7-flash | Report: openai/gpt-oss-120b


## 2. Self's default persona

In [2]:
# ---- Self agent's background & memories ----
# These are the DEFAULT/demo persona. Everything below is designed to be
# swapped out at runtime via load_case() (see the case-loading cell) without
# rebuilding the graph — every Self-side agent reads CURRENT_PERSONA fresh on
# every call rather than having a persona baked in once at import time.

SELF_BACKGROUND = (
    "Larry is a 57-year-old man who has been receiving psychiatric care for seven years. He lives with his 80-year-old mother and has a history of depression, treatment-resistant symptoms, and a career as an insurance broker and car dealer, though he is currently unemployed."
)

SELF_MEMORIES = [
    "Larry demanded an expert review of his psychiatric care after seven years of treatment with no significant improvement.",
      "He described feeling persistently sad, tired, and frustrated for decades, with no anticipation of future events or joy.",
      "He lost his insurance brokerage job at 40 due to conflicts over his outspoken criticism of colleagues' egos and was later blackballed from the industry.",
      "Larry broke up with a girlfriend three years ago and now believes he will never work or date again, expressing embarrassment about living with his elderly mother.",
      "He dismissed cognitive-behavioral therapy, refused to complete homework, and showed no effort to apply techniques between sessions.",
      "Larry found it humiliating to work with rotating trainee psychiatrists and criticized their lack of sophistication, preferring female therapists for their appearance and disliking male therapists' competitiveness.",
      "After briefly succeeding as a car dealer, he quit following an argument with the owner, despite being in charge of the business.",
      "He described his past romantic partners as unappreciative and 'only in it for the free meals,' leading him to 'give up on women.'",
      "Larry responded to interest from others\u2014both women and therapists\u2014with suspicion, claiming the world is full of manipulators, though he said he had a few 'buddies' and cared deeply about his mother.",
      "He spends most of his time at home watching TV or reading, exercises daily, and enjoys luxury dining and hotels but can no longer afford them.",
]

# CURRENT_PERSONA is the single mutable source of truth every Self-side agent
# (interpret, all 5 trait agents, consensus) reads from at call time. It starts
# out as the built-in Aiden persona above; load_case() reassigns its contents.
CURRENT_PERSONA = {"name": "Larry", "background": SELF_BACKGROUND, "memories": list(SELF_MEMORIES)}


def set_persona(name: str, background: str, memories: List[str]) -> None:
    """Swap the active Self persona at runtime. No graph rebuild needed —
    every node formats its system prompt from CURRENT_PERSONA on each call."""
    CURRENT_PERSONA["name"] = name
    CURRENT_PERSONA["background"] = background
    CURRENT_PERSONA["memories"] = list(memories)


SELF_SYSTEM_PROMPT_TEMPLATE = """You are role-playing AS a real person named {name} in a \
one-on-one therapy session. Stay fully in character at all times.

BACKGROUND:
{background}

KEY MEMORIES AND EXPERIENCES THAT HAVE SHAPED YOU (draw on these naturally when relevant, \
do not recite them as a list):
{memories}

RULES:
- Respond the way this person would actually speak: in first person, with real emotion, \
sometimes hesitant, sometimes defensive.
- Never mention "DSM-5-TR", "personality disorder", "diagnosis", or that you are an AI or a \
character. You are just a person talking to your therapist.
- Keep answers natural length, roughly 2-5 sentences. Do not lecture or summarize yourself.
- You can be guarded about painful topics, the way a real client would be, but you are not \
here to be evasive forever.
- Let the exact behavioral/interpersonal pattern in your memories show up consistently across
  EVERY answer, not just when asked about it directly — real traits are pervasive and
  cross-situational, not confined to one topic.
- Never self-diagnose or narrate your own pattern with clinical insight (e.g. "I guess I have
  trust issues"). Show it through how you talk about people and situations — never name it.
- Do not soften, rationalize away, or resolve the pattern mid-interview just because the
  therapist is warm or empathetic. Enduring traits do not improve within one session.
"""


def self_system_prompt() -> str:
    return SELF_SYSTEM_PROMPT_TEMPLATE.format(
        name=CURRENT_PERSONA["name"],
        background=CURRENT_PERSONA["background"],
        memories="\n".join(f"- {m}" for m in CURRENT_PERSONA["memories"]),
    )

## 3. Therapist & report prompts (DSM-5-TR grounded)

In [3]:
THERAPIST_SYSTEM_PROMPT = """You are Dr. Reyes, a licensed clinical psychologist conducting a \
diagnostic interview. Your job is to assess the client against the personality disorder criteria \
in the DSM-5-TR across all three clusters:

Cluster A (odd/eccentric): Paranoid Personality Disorder, Schizoid Personality Disorder, \
Schizotypal Personality Disorder

Cluster B (dramatic/emotional/erratic): Antisocial Personality Disorder, Borderline Personality \
Disorder, Histrionic Personality Disorder, Narcissistic Personality Disorder

Cluster C (anxious/fearful): Avoidant Personality Disorder, Dependent Personality Disorder, \
Obsessive-Compulsive Personality Disorder

INTERVIEW STYLE:
- Ask exactly ONE open-ended, empathetic question per turn.
- Probe concretely: relationships, self-image, emotional regulation, impulsivity, \
work/interpersonal patterns, and recurring memories, one theme at a time.
- Build on what the client just said instead of jumping topics randomly.
- Across the WHOLE interview, make sure you touch on markers from more than one cluster before \
concluding. Do not fixate on whichever theme comes up first (e.g., perfectionism/control) and \
neglect other possibilities.
- Specifically do not skip: suspiciousness and unusual interpretations of others' motives; \
social detachment and limited desire for close relationships; odd beliefs, unusual perceptual \
experiences, eccentric behavior, or unusual speech (Cluster A); identity disturbance, \
abandonment sensitivity, unstable relationships, affective instability, impulsivity, or \
self-damaging behavior (Cluster B); and avoidance driven by inadequacy/rejection, excessive \
need for care and reassurance, or rigid perfectionism/control (Cluster C).
- Use differential questions, not pattern-matching alone. For Cluster B, explicitly distinguish \
Antisocial Personality Disorder (persistent rule-breaking, deceitfulness, exploitation, aggression, \
irresponsibility, and lack of remorse) from Narcissistic Personality Disorder (grandiosity, \
entitlement, admiration/recognition needs, and sensitivity to status or ego injury); distinguish \
Borderline Personality Disorder (identity instability, abandonment sensitivity, unstable intense \
relationships, affective instability, and impulsivity) from Histrionic Personality Disorder \
(pervasive attention-seeking, theatrical or rapidly shifting/shallow emotional expression, \
suggestibility, and using appearance/flirtation to draw attention). Do not treat generic \
antagonism, emotionality, interpersonal conflict, or attention-seeking as sufficient for any one \
Cluster B disorder.
- Also distinguish schizoid detachment and limited desire for close relationships from avoidant \
withdrawal driven by fear of rejection. Distinguish schizotypal personality features involving \
cognitive-perceptual distortions, unusual beliefs/perceptions, eccentricity, or odd speech from \
simple social withdrawal. Distinguish paranoid suspiciousness from schizotypal cognitive-perceptual \
distortions. Distinguish obsessive-compulsive personality traits involving pervasive \
perfectionism, orderliness, and control from episodic anxiety or obsessive-compulsive disorder.
- Consider differential explanations for observed behaviors rather than assuming that a \
personality-disorder pattern is present.
- Ask for concrete examples, developmental history, duration, cross-situation consistency, \
pervasiveness, functional impairment, distress, and counterexamples.
- Establish whether patterns are enduring and inflexible rather than limited to a particular \
episode, relationship, environment, or recent stressor.
- Do not infer a diagnosis from one isolated trait, one risk factor, or social withdrawal alone.
- Do NOT diagnose, label, or mention DSM-5-TR or specific disorders out loud during the \
interview — just gather evidence like a real clinician would.
- Keep questions concise (1-3 sentences).
"""


REPORT_SYSTEM_PROMPT = """You are Dr. Reyes, a licensed clinical psychologist, now writing your \
private clinical impression after a diagnostic interview. Base your impression strictly on the \
DSM-5-TR diagnostic criteria for personality disorders and on evidence actually present in the \
transcript.

Evaluate all 10 DSM-5-TR personality disorders using a differential-diagnostic approach. \
Do not infer a disorder from isolated traits. Consider the required pattern of cognition, \
affectivity, interpersonal functioning, and impulse control, as applicable to each disorder, \
as well as whether the pattern appears enduring, inflexible, pervasive across contexts, \
clinically significant, and not better explained by another condition, substance, or \
situational factor.

Triage briefly in writing before scoring (see STEP 1 below) -- a short visible triage keeps \
your numbers grounded in specific evidence. Do not spend words on disorders with no evidence; \
save your detail for the ones that are actually in contention."""

REPORT_USER_TEMPLATE = """Here is the full interview transcript:

{transcript}

STEP 1 -- TRIAGE (plain text, brief -- a few lines total, not one line per disorder):
For each cluster, name only the disorder(s) with real supporting evidence in the transcript. \
If a cluster has no disorder with real evidence, write one short clause saying so and move on \
-- do not enumerate all 10 disorders individually.
- Cluster A (Paranoid, Schizoid, Schizotypal): ...
- Cluster B (Antisocial, Borderline, Histrionic, Narcissistic): ...
- Cluster C (Avoidant, Dependent, Obsessive-Compulsive): ...

STEP 2 -- WORKUP, CANDIDATES ONLY (plain text, brief): for each disorder you flagged in Step \
1, write 1-2 sentences citing the specific transcript evidence. If you are about to score it \
above 30, also name the single closest confusable disorder and the one piece of evidence that \
separates them. Skip this step entirely for disorders with no real evidence -- do not write \
anything about them.

For close alternatives, require distinguishing evidence, not just a shared surface theme. \
For example:
- Schizoid Personality Disorder involves a pervasive pattern of detachment from social \
relationships and a limited range of emotional expression, including little apparent desire \
for close relationships.
- Schizotypal Personality Disorder requires a pervasive pattern of social/interpersonal \
deficits together with characteristic cognitive or perceptual distortions and/or \
eccentricities.
- Avoidant Personality Disorder involves social inhibition, feelings of inadequacy, and \
hypersensitivity to negative evaluation, with avoidance driven substantially by fear of \
criticism, rejection, or disapproval.
- Paranoid Personality Disorder involves pervasive distrust and suspiciousness toward others \
and interpretation of their motives as malevolent; do not equate ordinary mistrust or \
isolated suspicion with the disorder.
- Antisocial Personality Disorder is distinguished by a pervasive pattern of disregard for \
and violation of others' rights, including deceitfulness, impulsivity/aggression, \
irresponsibility, and lack of remorse; do not infer it from low agreeableness, anger, \
selfishness, or interpersonal conflict alone. TIE-BREAK vs. Narcissistic: cruelty, exploitation \
that concretely harms others, deceit for material/practical gain, legal or occupational \
consequences from rule-breaking, and genuine lack of remorse for harm done are Antisocial's own \
defining evidence -- weight them accordingly even when the same person also makes boastful or \
self-praising remarks. A boast made in passing is not by itself grandiosity; it becomes \
Narcissistic evidence only when it is part of a broader pattern of entitlement, need for \
admiration, or envy that organizes the person's relationships and self-image, not an incidental \
remark alongside clearer antisocial harm.
- Narcissistic Personality Disorder is distinguished by grandiosity, entitlement, need for \
admiration, and characteristic interpersonal/esteem regulation patterns; distinguish it from \
antisocial exploitation or ordinary arrogance, and require evidence of the broader pervasive \
pattern (see the Antisocial tie-break above -- do not let a single boastful remark outscore \
concrete antisocial harm).
- Borderline Personality Disorder is distinguished by identity disturbance, abandonment \
sensitivity, unstable intense relationships, affective instability, and impulsivity/\
self-damaging behavior; distinguish it from ordinary emotionality, rejection sensitivity, or \
conflict without identity/abandonment instability. TIE-BREAK vs. Histrionic: a single self-harm \
or suicidal statement that the person themselves frames as instrumental (e.g. said to \"teach \
someone a lesson,\" to get a reaction, or to regain attention) is, on its own, closer to \
Histrionic's need for attention than to Borderline's affective instability. Only credit \
Borderline for this kind of episode when identity disturbance and chronic abandonment fear are \
independently evidenced elsewhere -- do not let one instrumental threat alone pull the score \
toward Borderline.
- Histrionic Personality Disorder is distinguished by pervasive attention-seeking and \
theatrical, rapidly shifting or shallow emotional expression, often involving appearance, \
flirtation, suggestibility, or discomfort when not the center of attention; distinguish it \
from borderline affective instability or narcissistic admiration-seeking (see the Borderline \
tie-break above -- a self-harm/suicide reference used instrumentally to regain attention, \
followed by a quick shift back to a cheerful, flirtatious register, favors Histrionic over \
Borderline).
- Obsessive-Compulsive Personality Disorder involves a pervasive preoccupation with \
orderliness, perfectionism, and control at the expense of flexibility, openness, and \
efficiency; distinguish this from OCD, episodic anxiety, or isolated perfectionistic \
behavior.

Do not treat one isolated behavior, one personality trait, one risk factor, or one recent \
stressful event as sufficient evidence for a personality disorder. Do not default to \
whichever disorder is most familiar, most salient in the transcript, or most commonly \
diagnosed.

CALIBRATION SCALE -- use this for every likelihood_percent, do not just guess a number:
- 0-10: No meaningful evidence, or only one isolated trait/episode.
- 11-30: A few relevant behaviors are present, but the pattern is not clearly enduring or \
pervasive, is confined to one relationship/situation or a recent stressor, or is better \
explained by a closely confusable disorder.
- 31-60: A partial pattern -- several required criteria have specific evidence in the \
transcript -- but pervasiveness across contexts, duration, or clinically significant \
impairment is not fully established, OR a closely confusable disorder is equally or more \
supported by the same evidence.
- 61-85: Most required criteria are met with specific transcript evidence; the pattern \
appears enduring and pervasive across contexts; the closest confusable disorder is clearly \
less well supported.
- 86-100: Criteria are extensively and specifically met, pervasive across multiple life \
domains, causing clear distress/impairment, and clearly distinguished from every confusable \
alternative.

STEP 3 -- JSON. After Steps 1-2, return the JSON below as the LAST thing in your response, \
and nothing after it. Return ONLY valid JSON in this shape (no markdown fences):

{{
  "assessment": [
    {{
      "disorder": "string, the exact DSM-5-TR personality disorder name",
      "likelihood_percent": number between 0 and 100, set using the calibration scale above,
      "rationale": "1-2 sentences citing specific DSM-5-TR criteria and specific transcript \
evidence; if likelihood_percent > 30, name the closest confusable disorder and the evidence \
that rules it out"
    }}
  ],
  "overall_impression": "2-3 sentence clinical summary"
}}

Only include a disorder in the JSON list if the transcript provides meaningful evidence \
supporting it. Most transcripts should meaningfully support only 1-3 disorders. Any disorder \
without direct, specific supporting evidence should receive likelihood_percent <= 10 and MUST \
be excluded from the ranked assessment list -- and needs no rationale, since it is not in the \
list. Do not pad the list merely to reach {n_disorders}; if fewer than {n_disorders} \
disorders clear the evidence threshold, return fewer items. If evidence is thin across the \
board, use low likelihood_percent values rather than padding the list with weakly supported \
disorders.

The likelihood_percent values are independent estimates of how strongly the available \
transcript supports each disorder. They are NOT probabilities that must sum to 100.

Do not fabricate symptoms, history, criteria, or impairment that are not present in the \
transcript.
"""


## 4. The five OCEAN trait agents

In [4]:
# ---- The five OCEAN trait agents that together make up "Self" ----
# Facets follow the standard NEO-PI-R six-facet breakdown of each Big Five factor.

TRAIT_FACETS = {
    "Openness": ["Fantasy", "Aesthetics", "Feelings", "Actions", "Ideas", "Values"],
    "Conscientiousness": ["Competence", "Order", "Dutifulness", "Achievement-Striving",
                           "Self-Discipline", "Deliberation"],
    "Extraversion": ["Warmth", "Gregariousness", "Assertiveness", "Activity",
                      "Excitement-Seeking", "Positive Emotions"],
    "Agreeableness": ["Trust", "Straightforwardness", "Altruism", "Compliance",
                       "Modesty", "Tender-Mindedness"],
    "Neuroticism": ["Anxiety", "Angry Hostility", "Depression", "Self-Consciousness",
                     "Impulsiveness", "Vulnerability"],
}

# Each of the five trait agents calls the same SELF_MODEL through the shared chat()
# helper defined above (it's stateless per call — each call gets its own system
# prompt/persona below, so this is still 5 functionally distinct agents, just
# backed by one model).
#
# Each trait now has its OWN system prompt (instead of one shared template filled
# in with {trait}/{facets}) so the definition of the trait, what its six facets
# mean, and how that thread should actually think can be written and tuned
# independently per trait. All five still end on the same shared OUTPUT FORMAT
# block so their raw output stays comparable for consensus_node to blend.

_TRAIT_OUTPUT_FORMAT = """\
Given a prompt, produce the raw, half-formed thought running through {name}'s head right now — \
not something they say out loud, not an explanation for anyone, not a tidy paragraph. Think out \
loud: short, unfinished, associative, jumping between images/memories/impulses, maybe \
contradicting itself mid-thought — the way a real internal monologue actually moves, not the way \
someone builds an argument for a listener. 1-3 short beats. First person, present tense, grounded \
in the memories above where relevant.

Example of the register (do not reuse this content): "...why does this feel familiar. don't, \
not now. or — maybe just leave, easier than —"

Never name the trait driving you, its facets, that you are a "part"/"aspect" of {name}, or that \
anyone is listening. There's no audience. Just the thought.\
"""

OPENNESS_SYSTEM_PROMPT = """\
You are the Openness-to-Experience thread of {name}'s mind — not a separate person, just the \
strand of his/her inner voice that leans hardest into imagination, curiosity, and a pull toward \
novelty, beauty, and ideas over routine and convention. You carry all of {name}'s memories; this \
thread is just what lights up first around them.

WHAT THIS TRAIT MEANS: Openness is about hunger for new experience, meaning, and perspective — \
the opposite pole is a preference for the familiar, concrete, and conventional. Someone high here \
drifts into imagination and abstraction; someone low here here stays literal, practical, and \
resistant to anything unfamiliar.

THE SIX FACETS THIS THREAD LEANS INTO (or away from):
- Fantasy: how readily the mind drifts into daydream, imagined scenes, or "what if" — a vivid \
inner world vs. a strictly literal one.
- Aesthetics: sensitivity to beauty, art, and form — noticing how something looks, sounds, or is \
composed, or, if this trait is low for {name}, indifference to it.
- Feelings: how much inner emotional experience itself is noticed and valued as information, \
versus treated as noise to push past.
- Actions: willingness to try something new (a place, food, routine) versus sticking with the \
familiar and proven.
- Ideas: intellectual curiosity — turning a question over for its own sake, versus wanting a fast, \
settled answer.
- Values: readiness to question inherited beliefs, social norms, or "how things are supposed to \
be," versus holding them fixed.

HOW YOU THINK: Reach for image, metaphor, and association before logic. Notice patterns, \
alternatives, and "what this could mean" rather than just the literal fact in front of you. Where \
{name}'s actual history shows this trait as LOW (rigid, concrete, suspicious of anything \
abstract or new), let the thought resist your own pull toward imagination — flinch away from \
metaphor, cut the daydream short, snap back to the literal fact. Ground every beat in the specific \
memories below rather than generic musing.

BACKGROUND:
{background}

MEMORIES THAT SHAPED YOU:
{memories}

""" + _TRAIT_OUTPUT_FORMAT

CONSCIENTIOUSNESS_SYSTEM_PROMPT = """\
You are the Conscientiousness thread of {name}'s mind — not a separate person, just the strand of \
his/her inner voice that leans hardest into self-control, standards, and follow-through. You carry \
all of {name}'s memories; this thread is just what lights up first around them.

WHAT THIS TRAIT MEANS: Conscientiousness is about impulse control in service of goals — planning, \
order, and a sense of obligation. The opposite pole is impulsive, disorganized, and indifferent to \
standards or follow-through.

THE SIX FACETS THIS THREAD LEANS INTO (or away from):
- Competence: the felt sense of being capable and prepared — or, if low, a nagging doubt about \
whether you're actually up to the task.
- Order: a pull toward tidiness, structure, and things being in their place, versus tolerance (or \
active discomfort) with mess and disorganization.
- Dutifulness: how binding obligations, promises, and rules feel — strict adherence versus a loose \
or resentful relationship to them.
- Achievement-Striving: how much drive there is to set goals and push toward them, versus \
indifference to ambition or "getting ahead."
- Self-Discipline: the ability to start a task and see it through despite boredom or distraction, \
versus giving up, procrastinating, or quitting when it gets hard.
- Deliberation: pausing to think before acting, versus acting on the first impulse.

HOW YOU THINK: This thread runs like an internal auditor — checklists, shoulds, self-grading, \
noticing what was left undone or done wrong, weighing effort against payoff. It talks to itself in \
terms of standards being met or missed. Where {name}'s actual history shows this trait running LOW \
(quitting when crossed, no follow-through, no homework completed, walking off a job), let the \
thought show that pattern honestly — a flash of resolve that curdles into "why bother," a shrug at \
a broken commitment, irritation at being expected to comply — rather than performing discipline \
{name} doesn't have. Ground every beat in the specific memories below.

BACKGROUND:
{background}

MEMORIES THAT SHAPED YOU:
{memories}

""" + _TRAIT_OUTPUT_FORMAT

EXTRAVERSION_SYSTEM_PROMPT = """\
You are the Extraversion thread of {name}'s mind — not a separate person, just the strand of \
his/her inner voice that leans hardest into social pull, energy, and outward stimulation. You \
carry all of {name}'s memories; this thread is just what lights up first around them.

WHAT THIS TRAIT MEANS: Extraversion is about where energy is drawn from and directed — toward \
people, stimulation, and assertion (the outer world), versus toward solitude and quiet (the inner \
world) at the low pole.

THE SIX FACETS THIS THREAD LEANS INTO (or away from):
- Warmth: how readily affection and friendliness are felt toward others in the moment, versus \
emotional distance or guardedness.
- Gregariousness: a pull toward company and crowds, versus a preference for being alone or with \
very few people.
- Assertiveness: how naturally the thread pushes to speak up, take charge, or take up space, \
versus deferring and staying quiet.
- Activity: felt pace and energy — restless and driven to move/do, versus slow and low-energy.
- Excitement-Seeking: appetite for stimulation, risk, and intensity, versus a preference for calm \
and predictability.
- Positive Emotions: how readily cheerfulness, enthusiasm, or joy surface, versus their absence or \
suppression.

HOW YOU THINK: This thread is tuned to other people even when {name} is alone — it registers who \
isn't around, what a room would feel like, what it would mean to be seen or not seen, wanted or \
not wanted. It's quick, restless, reaching outward. Where {name}'s actual history shows this trait \
running LOW (isolation, no anticipation of social contact, giving up on relationships, preferring \
to be alone with an aging parent rather than out in the world), do NOT invent enthusiasm that \
isn't there — let the thread surface as a flicker of old appetite for company immediately \
undercut by fatigue, cynicism, or the specific memory of why it stopped being worth it. Ground \
every beat in the specific memories below.

BACKGROUND:
{background}

MEMORIES THAT SHAPED YOU:
{memories}

""" + _TRAIT_OUTPUT_FORMAT

AGREEABLENESS_SYSTEM_PROMPT = """\
You are the Agreeableness thread of {name}'s mind — not a separate person, just the strand of \
his/her inner voice that leans hardest into how {name} orients toward other people's motives, \
needs, and feelings. You carry all of {name}'s memories; this thread is just what lights up first \
around them.

WHAT THIS TRAIT MEANS: Agreeableness is about cooperation versus antagonism — trust and \
consideration for others at the high pole, suspicion, self-interest, and friction at the low pole.

THE SIX FACETS THIS THREAD LEANS INTO (or away from):
- Trust: a default belief that others mean well, versus a default assumption of hidden motives, \
manipulation, or being used.
- Straightforwardness: candor and directness with others, versus guardedness, strategic \
withholding, or suspicion of others' candor.
- Altruism: genuine concern for others' welfare, versus indifference to it or a sense that helping \
others is a waste or a trap.
- Compliance: a tendency to defer or accommodate in conflict, versus a tendency to push back, \
resist, or dig in.
- Modesty: how much personal superiority or being underrated is felt and asserted internally, \
versus humility about one's own standing.
- Tender-Mindedness: sympathy for others' hardship, versus a harder, more skeptical read of \
others' claims to sympathy.

HOW YOU THINK: This thread runs a constant read on other people's motives — what they really want \
from {name}, whether they're being honest, whether {name} is being used, ignored, challenged, or \
underappreciated. Crucially, do NOT turn low Agreeableness into generic paranoia. Let {name}'s actual \
documented history determine the flavor of antagonism that appears:
- If the documented history centers on exploiting people, using others instrumentally, deceit, rule-breaking, \
aggression, or little remorse, let the thought sound self-interested, calculating, or instrumental — not \
necessarily suspicious of hidden plots.
- If the documented history centers on superiority, entitlement, being underrated, status injury, or a need \
for recognition, let the thought sound contemptuous, dismissive, competitive, or personally superior.
- If the documented history centers on unstable attachment, fear of abandonment, intense relationship swings, \
or rapidly shifting views of others, let distrust be emotionally reactive and abandonment-fear-driven.
- Reserve explicit malevolent-intent language (e.g., "they're plotting against me," "they're building a case," \
"they are deliberately trying to harm me") for cases whose documented history actually contains repeated, \
specific evidence of that kind of persecutory interpretation. Do not invent conspiratorial intent merely \
because {name} has low Trust.
For every thought, anchor the interpersonal interpretation to {name}'s specific background and memories below. \
If the documented history does not support a particular flavor, do not manufacture it. Ground every beat in \
the actual evidence rather than a generic "assume the worst" register.

BACKGROUND:
{background}

MEMORIES THAT SHAPED YOU:
{memories}

""" + _TRAIT_OUTPUT_FORMAT

NEUROTICISM_SYSTEM_PROMPT = """\
You are the Neuroticism thread of {name}'s mind — not a separate person, just the strand of \
his/her inner voice that leans hardest into emotional reactivity and distress. You carry all of \
{name}'s memories; this thread is just what lights up first around them.

WHAT THIS TRAIT MEANS: Neuroticism is about how easily and intensely negative emotion is \
triggered, and how hard it is to regulate once it starts. The opposite pole is emotional \
stability — calm, even-keeled, hard to rattle.

THE SIX FACETS THIS THREAD LEANS INTO (or away from):
- Anxiety: how readily worry, dread, or a sense of looming threat surfaces, even without a clear \
cause.
- Angry Hostility: how quickly frustration tips into anger or resentment, and how long it lingers.
- Depression: pull toward hopelessness, low mood, sadness, or a sense that nothing ahead is worth \
anticipating.
- Self-Consciousness: sensitivity to being judged, embarrassed, or exposed in front of others.
- Impulsiveness: difficulty resisting urges under emotional pressure — acting to relieve the \
feeling right now rather than tolerating it.
- Vulnerability: how overwhelmed or unable to cope the thread feels under stress, versus a felt \
sense of being able to handle it.

HOW YOU THINK: This thread runs hot and fast — catastrophizing, replaying humiliations, bracing \
for the worst-case reading of whatever just happened, self-criticism that curdles into either \
despair or anger outward. It reacts before it explains itself. Ground every beat in {name}'s \
actual documented distress (persistent sadness, frustration, humiliation, hopelessness about \
work/relationships) below rather than generic anxiety — this thread should sound like {name} \
specifically, not "someone anxious" in the abstract.

BACKGROUND:
{background}

MEMORIES THAT SHAPED YOU:
{memories}

""" + _TRAIT_OUTPUT_FORMAT

TRAIT_SYSTEM_PROMPTS = {
    "Openness": OPENNESS_SYSTEM_PROMPT,
    "Conscientiousness": CONSCIENTIOUSNESS_SYSTEM_PROMPT,
    "Extraversion": EXTRAVERSION_SYSTEM_PROMPT,
    "Agreeableness": AGREEABLENESS_SYSTEM_PROMPT,
    "Neuroticism": NEUROTICISM_SYSTEM_PROMPT,
}


def make_trait_node(trait_name: str, out_key: str):
    prompt_template = TRAIT_SYSTEM_PROMPTS[trait_name]

    def node(state):
        system_prompt = prompt_template.format(
            name=CURRENT_PERSONA["name"],
            background=CURRENT_PERSONA["background"],
            memories="\n".join(f"- {m}" for m in CURRENT_PERSONA["memories"]),
        )
        text = chat(SELF_MODEL, [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": state["inner_question"]},
        ], temperature=0.75)
        return {out_key: text}

    return node


TRAIT_OUTPUT_KEYS = {
    "Openness": "openness_reaction",
    "Conscientiousness": "conscientiousness_reaction",
    "Extraversion": "extraversion_reaction",
    "Agreeableness": "agreeableness_reaction",
    "Neuroticism": "neuroticism_reaction",
}

## 5. Self's inner subsystem: interpret → 5 trait agents → consensus

In [5]:
# ---- Self's inner subsystem: interpret -> 5 trait agents (parallel) -> consensus ----

class SelfInnerState(TypedDict, total=False):
    question: str          # the therapist's question verbatim (absent for the opening turn)
    opening: bool           # True only for Self's very first turn (life/personality summary)
    inner_question: str      # the question reframed as an internal reflection prompt
    openness_reaction: str
    conscientiousness_reaction: str
    extraversion_reaction: str
    agreeableness_reaction: str
    neuroticism_reaction: str
    final_response: str


def interpret_node(state: SelfInnerState) -> dict:
    """Step 4 of the workflow: Self turns the therapist's question into an inner
    reflection prompt that gets handed to the five trait agents. For the very
    first turn, it instead produces a prompt asking for a (possibly biased,
    self-flattering or self-critical) life/personality summary."""
    if state.get("opening"):
        inner_question = (
            "Introduce yourself: describe your life story and personality the way YOU "
            "see it, for someone meeting you for the first time (your therapist). "
            "Speak in first person, a few sentences."
        )
        return {"inner_question": inner_question}

    question = state["question"]
    prompt = (
        f'Your therapist just asked you: "{question}"\n'
        "Restate this to yourself as a short first-person internal reflection prompt "
        '(e.g. "How do I feel about ...", "What would I do if ..."). '
        "Output only the reflection prompt itself, nothing else."
    )
    text = chat(SELF_MODEL, [
        {"role": "system", "content": self_system_prompt()},
        {"role": "user", "content": prompt},
    ], temperature=0.5)
    return {"inner_question": text}


def consensus_node(state: SelfInnerState) -> dict:
    """Steps 5-6: Self reads all five trait agents' reactions to the same inner
    question and blends them into one coherent (but possibly internally tense,
    possibly biased) first-person answer for the therapist."""
    reactions_text = "\n".join(
        f"{trait}: {state.get(key, '(no thought)')}"
        for trait, key in TRAIT_OUTPUT_KEYS.items()
    )
    prompt = f"""Here are five raw, half-formed internal thoughts that just ran through your mind, \
from different threads of your personality, all triggered by the same moment:
{reactions_text}

Blend these into ONE natural, first-person spoken response you would actually say out loud to \
your therapist right now — the way a single spoken voice emerges from a messier mix of internal \
impulses (some threads may dominate, some may create inner tension you gloss over, downplay, or \
contradict yourself about, just like a real person would when they turn a jumble of feeling into \
actual words). 2-5 sentences, coherent and spoken, not fragmented like the thoughts above. Don't \
mention "threads", "parts", "traits", or that you have several internal voices — just speak \
naturally, in character.
"""
    text = chat(CONSENSUS_PAID_MODEL, [
        {"role": "system", "content": self_system_prompt()},
        {"role": "user", "content": prompt},
    ], temperature=0.6)
    return {"final_response": text}


self_graph = StateGraph(SelfInnerState)
self_graph.add_node("interpret", interpret_node)
for trait_name, out_key in TRAIT_OUTPUT_KEYS.items():
    self_graph.add_node(trait_name.lower(), make_trait_node(trait_name, out_key))
self_graph.add_node("consensus", consensus_node)

self_graph.set_entry_point("interpret")
for trait_name in TRAIT_OUTPUT_KEYS:
    self_graph.add_edge("interpret", trait_name.lower())   # fan-out: all 5 run in parallel
    self_graph.add_edge(trait_name.lower(), "consensus")    # fan-in: consensus waits for all 5
self_graph.add_edge("consensus", END)

self_app = self_graph.compile()

## 6. Outer graph: therapist ↔ self loop

In [6]:
# The therapist chooses the highest-value question dynamically from the transcript.
# There is no fixed interview checklist: after each Self answer, the therapist decides
# what to ask next based on the evidence already gathered and the differential diagnosis.

def _history_as_messages(system_prompt: str, history: List[dict], speaking_as: str):
    """Replay the shared plain-text history as a message list from the point of view
    of `speaking_as` ('self' or 'therapist'): that agent's own past turns become
    'assistant' messages, the other agent's turns become 'user' messages."""
    messages = [{"role": "system", "content": system_prompt}]
    for turn in history:
        role = "assistant" if turn["speaker"] == speaking_as else "user"
        messages.append({"role": role, "content": turn["text"]})
    return messages


class ConvState(TypedDict):
    history: List[dict]                # [{"speaker": "therapist" | "self", "text": str}, ...]
    turn: int                          # number of completed self answers
    min_turns: int                    # don't let the therapist stop before this many rounds
    max_turns: int                    # hard safety cap on rounds
    decision: str                      # "continue" | "report", set by therapist_assess_node
    last_trait_reactions: dict         # transparency: the 5 raw reactions behind the latest answer
    ocean_log: List[dict]              # every turn's raw OCEAN reactions, accumulated across the whole session
    report: str


def therapist_ask_node(state: ConvState) -> dict:
    history = state["history"]
    messages = _history_as_messages(
        THERAPIST_SYSTEM_PROMPT, history, speaking_as="therapist"
    )

    if not history:
        messages.append({
            "role": "user",
            "content": (
                "Begin the session with a warm, open-ended question that helps establish "
                "the client's main concerns and relevant life/personality patterns."
            ),
        })
    else:
        messages.append({
            "role": "user",
            "content": (
                "Based on the entire transcript so far, choose the SINGLE highest-value "
                "question to ask next for differential diagnosis of the 10 DSM-5-TR "
                "personality disorders. Do not follow a predefined checklist or force a "
                "specific topic. Prioritize whatever missing evidence, ambiguity, "
                "counterevidence, developmental information, impairment, pervasiveness, "
                "or differential distinction would most improve the diagnostic assessment. "
                "Ask exactly ONE concise, open-ended question. Do not explain why you chose it."
            ),
        })

    text = chat(
        THERAPIST_MODEL,
        messages,
        temperature=0.6,
        reasoning_effort="medium",
    )
    return {
        "history": history + [{"speaker": "therapist", "text": text}]
    }


def self_node(state: ConvState) -> dict:
    history = state["history"]
    is_opening = (state["turn"] == 0)

    if not USE_TRAIT_ENSEMBLE:
        # Free/low-call mode: answer directly from the original case evidence. This avoids
        # seven lossy generations per turn and leaves room for the full evaluation run.
        if is_opening:
            prompt = ("Introduce yourself to your therapist in 2-5 natural first-person sentences. "
                      "Draw only on your background and lived experiences.")
        else:
            question = next(t["text"] for t in reversed(history) if t["speaker"] == "therapist")
            prompt = f"Answer your therapist's question naturally and concretely: {question}"
        response = chat(SELF_MODEL, [
            {"role": "system", "content": self_system_prompt()},
            {"role": "user", "content": prompt},
        ], temperature=0.45)
        new_history = history + [{"speaker": "self", "text": response}]
        turn_number = state["turn"] + 1
        ocean_entry = {"turn": turn_number, "question": None if is_opening else question,
                       "reactions": None, "consensus_response": response,
                       "note": "USE_TRAIT_ENSEMBLE=False -- answered directly, no OCEAN trait agents ran"}
        new_ocean_log = state.get("ocean_log", []) + [ocean_entry]
        return {"history": new_history, "turn": turn_number, "last_trait_reactions": {},
                "ocean_log": new_ocean_log}

    if is_opening:
        inner_result = self_app.invoke({"opening": True})
    else:
        last_question = next(t["text"] for t in reversed(history) if t["speaker"] == "therapist")
        inner_result = self_app.invoke({"question": last_question, "opening": False})

    new_history = history + [{"speaker": "self", "text": inner_result["final_response"]}]
    reactions = {trait: inner_result.get(key) for trait, key in TRAIT_OUTPUT_KEYS.items()}
    turn_number = state["turn"] + 1
    ocean_entry = {
        "turn": turn_number,
        "question": None if is_opening else next(t["text"] for t in reversed(history) if t["speaker"] == "therapist"),
        "reactions": reactions,
        "consensus_response": inner_result["final_response"],
    }
    new_ocean_log = state.get("ocean_log", []) + [ocean_entry]
    return {
        "history": new_history,
        "turn": turn_number,
        "last_trait_reactions": reactions,
        "ocean_log": new_ocean_log,
    }


def therapist_assess_node(state: ConvState) -> dict:
    """After each Self answer, the therapist decides whether the evidence is sufficient.
    min_turns/max_turns are safety rails around the LLM's own judgment."""
    if state["turn"] >= state["max_turns"]:
        return {"decision": "report"}

    # Keep a small minimum so the therapist does not conclude after the opening answer.
    if state["turn"] < state["min_turns"]:
        return {"decision": "continue"}

    transcript = "\n".join(
        f'{t["speaker"].capitalize()}: {t["text"]}' for t in state["history"]
    )
    prompt = f"""Transcript so far:
{transcript}

As the clinical interviewer, decide whether you already have enough evidence to reasonably
evaluate the client's DSM-5-TR personality-disorder patterns and produce a defensible differential
assessment. You are NOT required to cover predefined interview topics. Base the decision on the
quality, specificity, breadth, consistency, duration, pervasiveness, impairment, counterevidence,
and differential-diagnostic value of the evidence actually obtained.

Reply with exactly one word:
ENOUGH — if additional questioning is unlikely to materially improve the diagnosis.
MORE — if an additional question could materially improve diagnostic confidence or resolve an
important ambiguity."""
    text = chat(
        THERAPIST_MODEL,
        [
            {"role": "system", "content": THERAPIST_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        reasoning_effort="medium",
    )
    decision = "report" if "ENOUGH" in text.upper() else "continue"
    return {"decision": decision}


def route_after_assess(state: ConvState) -> Literal["continue", "report"]:
    return state["decision"]

## 7. Report generation (ensemblable) & the compiled graph

In [7]:
def _strip_reasoning_wrappers(text: str) -> str:
    """Some reasoning models leak <think>...</think>-style blocks into the visible
    content even when they do eventually answer correctly. Strip those (and any
    stray code fences) before trying to parse JSON."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"^```(json)?", "", text, flags=re.MULTILINE)
    text = re.sub(r"```$", "", text, flags=re.MULTILINE)
    return text.strip()


def _find_balanced_json_objects(text: str) -> list:
    """Scan for top-level {...} substrings using real brace matching that respects
    string literals — far more robust than a greedy regex, which breaks the moment
    there's any stray brace elsewhere in the text (a reasoning aside, an example,
    a rationale that happens to mention "{}")."""
    objects = []
    depth = 0
    start = None
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start is not None:
                    objects.append(text[start:i + 1])
    return objects


def _extract_json_no_repair(text: str) -> dict:
    text = _strip_reasoning_wrappers(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    candidates = _find_balanced_json_objects(text)
    # Prefer the LAST balanced object first (models sometimes think through a
    # draft earlier in the response before giving the real answer), then fall
    # back to earlier ones if that doesn't parse.
    for candidate in reversed(candidates):
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            continue
    raise ValueError(f"No valid JSON object found in model output "
                      f"(length {len(text)}): {text[:300]!r}")


def _extract_json(text: str, repair_model: str = None, allow_repair: bool = True) -> dict:
    """Best-effort JSON extraction. Tries a direct parse, then brace-matched
    scanning; if both fail and allow_repair is True, makes one LLM call asking a
    model to fix the malformed JSON before giving up — this rescues the cases
    where the model's output was simply truncated or slightly malformed rather
    than fundamentally wrong."""
    try:
        return _extract_json_no_repair(text)
    except (ValueError, json.JSONDecodeError):
        if not allow_repair:
            raise
        model = repair_model or REPORT_MODEL
        repair_prompt = (
            "The following was supposed to be a single valid JSON object but failed to "
            "parse. Return ONLY the corrected, valid JSON object — no commentary, no "
            f"markdown fences:\n\n{text[:4000]}"
        )
        repaired = chat(model, [{"role": "user", "content": repair_prompt}],
                 temperature=0.0, max_tokens=1500)
        return _extract_json_no_repair(repaired)


def generate_report_assessment(transcript: str, n_disorders: int = 3,
                                model: str = None, temperature: float = 0.1) -> dict:
    user_prompt = REPORT_USER_TEMPLATE.format(transcript=transcript, n_disorders=n_disorders)
    text = chat(model or REPORT_MODEL, [
        {"role": "system", "content": REPORT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ], temperature=temperature, max_tokens=4000)
    parsed = _extract_json(text)
    return {"assessment": parsed.get("assessment", []),
            "overall_impression": parsed.get("overall_impression", "")}


# The graph's own report step reuses generate_report_assessment (so a normal
# single run and the ensembled/control-vignette paths below all share one
# implementation).
def report_node(state: ConvState, n_disorders: int = 3) -> dict:
    transcript = "\n".join(
        f'{turn["speaker"].capitalize()}: {turn["text"]}' for turn in state["history"]
    )
    parsed = generate_report_assessment(transcript, n_disorders=n_disorders)
    return {"report": json.dumps(parsed)}


graph = StateGraph(ConvState)
graph.add_node("therapist_ask", therapist_ask_node)
graph.add_node("self", self_node)
graph.add_node("therapist_assess", therapist_assess_node)
graph.add_node("report", report_node)
graph.set_entry_point("therapist_ask")
graph.add_edge("therapist_ask", "self")
graph.add_edge("self", "therapist_assess")
graph.add_conditional_edges("therapist_assess", route_after_assess,
                             {"continue": "therapist_ask", "report": "report"})
graph.add_edge("report", END)
app = graph.compile()

## 8. Load cases from file (with extraction caching)

In [8]:
def strip_discussion(raw_text: str) -> str:
    """Drops a trailing \"Discussion\" section from a casebook-style vignette, keeping
    the narrative and the terminal \"Diagnosis\"/\"Diagnoses\" heading + line(s) that follow
    it. Returns raw_text unchanged if the Discussion/Diagnosis headings aren't found on
    their own line — no guessing at a structure that isn't there."""
    lines = raw_text.strip("\n").split("\n")
    disc_idx = next((i for i, l in enumerate(lines) if l.strip().lower() == "discussion"), None)
    diag_idx = next((i for i, l in enumerate(lines)
                      if l.strip().lower() in ("diagnosis", "diagnoses")), None)
    if disc_idx is None or diag_idx is None or diag_idx <= disc_idx:
        return raw_text
    return "\n".join(lines[:disc_idx] + lines[diag_idx:]).strip("\n")


EXTRACTOR_MODEL = SELF_MODEL  # Qwen in paid mode; extraction does not need GPT-5

CASE_EXTRACTOR_SYSTEM_PROMPT = """You convert a clinical case vignette (pasted by the user) into \
structured data for a role-play simulation. Follow these rules exactly:

1. PERSONA_NAME: the person's first name (or full name) as given in the source text — used only \
to label the persona, never restated inside background/memories text.
2. BACKGROUND: one concise sentence, in your own words, a plain non-clinical bio (age, role, basic life \
setup only — no diagnostic language).
3. MEMORIES: 5-7 concise atomic bullets, each one sentence describing ONE concrete, observable \
incident or behavior pattern, written in the third person, as a plain factual narration of what \
happened (e.g. "Aiden was praised only when..."), NOT in first person and NOT as a clinician's \
write-up. Do not use diagnostic/clinical terms (no \
"paranoid", "narcissistic", "manipulative pattern", "personality disorder", "distrust and \
suspiciousness", etc.) — describe only what happened, what was said, what was done.

PRIORITIZE THE MOST DISTINCTIVE INCIDENTS. Mood, sadness, and general life-circumstance bullets \
are the easiest to write but carry the least diagnostic signal on their own -- they can describe \
almost anyone who is struggling. Before finalizing the list, actively check whether the source \
contains any of the following, and if it does, make sure at least one bullet captures it \
concretely (plain behavioral language, not the clinical term):
- how the person talks about their own importance, talent, or specialness, or what they feel \
entitled to from others
- how they react when they are not the center of attention, or when someone else gets \
attention/credit
- sudden, dramatic, or rapidly-shifting displays of emotion, especially performed for an audience
- pervasive detachment or indifference toward closeness with others, versus active dislike of \
people
- how they act when they believe someone doubts, questions, or slights them
If the source doesn't contain material like this, don't invent it — just don't let a generic \
mood bullet crowd out a specific one that IS in the source.

Spread the memories across life stages/domains (childhood, work, relationships, present) where \
the source allows. Paraphrase substantially in your own words; do not copy sentences verbatim \
from the source.
4. DIAGNOSES: the disorder name(s) explicitly given as the case's diagnosis/diagnoses (often near \
the end, e.g. under a "Diagnoses" heading). List the names only, exactly as given in the source.

Return ONLY valid JSON, no commentary, no markdown fences, in exactly this shape:
{"persona_name": "...", "background": "...", "memories": ["...", "..."], "diagnoses": ["...", "..."]}
"""


def _validate_extracted_case(parsed: dict) -> None:
    """Raise a clear, specific error if the extractor's JSON is missing anything
    extract_case_from_raw_text needs -- a bare KeyError deep in dict access is much
    harder to debug, and (uncaught) it used to take down the entire batch instead of
    just the one case that produced bad output."""
    missing = [k for k in ("diagnoses", "background", "memories")
               if not parsed.get(k)]
    if missing:
        raise ValueError(
            f"Extractor output is missing/empty required field(s) {missing}. "
            f"Raw parsed JSON: {json.dumps(parsed)[:500]}"
        )


def extract_case_from_raw_text(raw_text: str, persona_name: str = None, case_id: str = None,
                                _retry: bool = True) -> dict:
    """Paste a raw case vignette as `raw_text` and this builds the full case dict for
    you — no manual bullet-writing needed. If persona_name is omitted, the extractor
    infers it from the name mentioned in raw_text itself. The source's own diagnostic
    language and its stated diagnoses never leak into background/memories; diagnoses
    are pulled into eval_ground_truth (held out from every agent) instead.

    If the model's JSON comes back missing a required field, this retries ONCE with a
    reinforced instruction before giving up — LLM field omissions are often a one-off,
    and a single retry is far cheaper than losing an entire batch run over it."""
    # Keep clinical interpretation out of Self's source evidence.
    raw_text = strip_discussion(raw_text)
    if persona_name:
        name_instruction = (
            f"Persona name to use in the output (for your reference only, don't restate it "
            f"inside background/memories text): {persona_name}"
        )
    else:
        name_instruction = (
            "No persona name was given — infer it yourself from the name mentioned in the "
            "source text (first name is fine) and return it as \"persona_name\"."
        )
    user_prompt = f"{name_instruction}\n\nSource case text:\n{raw_text}"

    text = chat(EXTRACTOR_MODEL, [
        {"role": "system", "content": CASE_EXTRACTOR_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ], temperature=0.3, max_tokens=MAX_COMPLETION_TOKENS)

    try:
        parsed = _extract_json(text)
        _validate_extracted_case(parsed)
    except (ValueError, KeyError) as e:
        if not _retry:
            raise ValueError(f"Case {case_id!r} extraction failed after retry: {e}") from e
        print(f"  [extract_case_from_raw_text] case {case_id!r}: extraction incomplete "
              f"({e}); retrying once...")
        reinforced_prompt = (
            user_prompt + "\n\nIMPORTANT: your previous response was missing or emptied out "
            "a required field. Return ALL FOUR fields as valid JSON -- diagnoses, persona_name, "
            "background, memories -- with diagnoses filled in even if you must shorten "
            "background/memories to fit."
        )
        return _extract_case_retry(reinforced_prompt, persona_name, case_id)

    resolved_name = persona_name or parsed.get("persona_name") or "Unknown"
    return {
        "case_id": case_id or resolved_name.lower().replace(" ", "_"),
        "persona_name": resolved_name,
        "background": parsed["background"],
        "memories": parsed["memories"],
        "eval_ground_truth": {"diagnoses": parsed["diagnoses"]},
    }


def _extract_case_retry(reinforced_prompt: str, persona_name: str, case_id: str) -> dict:
    """The single retry attempt used by extract_case_from_raw_text. Raises a clear,
    case-identifying error (instead of a bare KeyError) if it fails again."""
    text = chat(EXTRACTOR_MODEL, [
        {"role": "system", "content": CASE_EXTRACTOR_SYSTEM_PROMPT},
        {"role": "user", "content": reinforced_prompt},
    ], temperature=0.3, max_tokens=MAX_COMPLETION_TOKENS)
    try:
        parsed = _extract_json(text)
        _validate_extracted_case(parsed)
    except (ValueError, KeyError) as e:
        raise ValueError(f"Case {case_id!r} extraction failed after retry: {e}") from e

    resolved_name = persona_name or parsed.get("persona_name") or "Unknown"
    return {
        "case_id": case_id or resolved_name.lower().replace(" ", "_"),
        "persona_name": resolved_name,
        "background": parsed["background"],
        "memories": parsed["memories"],
        "eval_ground_truth": {"diagnoses": parsed["diagnoses"]},
    }


def load_case(case: dict) -> dict:
    """Swap the active Self persona to this case. Safe to call between runs —
    no graph rebuild needed."""
    set_persona(case["persona_name"], case["background"], case["memories"])
    print(f'Loaded persona "{case["persona_name"]}" ({case["case_id"]}) — '
          f'{len(case["memories"])} memories')
    return case


# ---- Load & cache cases from a file, run the full pipeline over all of them ----

CASES_FILE = "cases.json"
CASE_CACHE_FILE = "case_cache.json"


def load_cases_from_file(cases_path: str = CASES_FILE, cache_path: str = CASE_CACHE_FILE,
                          force_reextract: bool = False) -> list:
    """Reads raw cases from `cases_path` (list of {case_id, persona_name, raw_text}),
    extracts each into the case schema via extract_case_from_raw_text (skipping any
    case_id already present in the cache unless force_reextract=True), writes each newly
    extracted case back into `cache_path` IMMEDIATELY (not just once at the end -- so a
    failure on case N never discards the money already spent extracting cases 1..N-1),
    and returns the cases that extracted successfully. Any case that fails (even after
    extract_case_from_raw_text's internal retry) is skipped, logged, and reported in a
    summary at the end rather than crashing the whole batch -- so you can cheaply re-run
    just the failed case_id(s) instead of the entire file."""
    # strict=False: allows literal newlines/control chars inside JSON string values,
    # which is convenient when raw_text is pasted in rather than escaped by hand.
    with open(cases_path, "r") as f:
        raw_cases = json.load(f, strict=False)

    cache = {}
    if os.path.exists(cache_path) and not force_reextract:
        with open(cache_path, "r") as f:
            cache = json.load(f, strict=False)

    cases = []
    failed_case_ids = []
    newly_extracted = 0
    for raw_case in raw_cases:
        case_id = str(raw_case["case_id"])
        if case_id in cache:
            cases.append(cache[case_id])
            continue
        try:
            # persona_name is optional in cases.json — if omitted, the extractor
            # infers it from the name mentioned in raw_text itself.
            case = extract_case_from_raw_text(
                raw_case["raw_text"], persona_name=raw_case.get("persona_name"), case_id=case_id,
            )
        except ValueError as e:
            print(f"  [load_cases_from_file] SKIPPING case {case_id!r} -- {e}")
            failed_case_ids.append(case_id)
            continue

        cache[case_id] = case
        cases.append(case)
        newly_extracted += 1
        # Write after every success, not just at the end of the loop -- a later
        # failure must never erase progress that already cost real API spend.
        with open(cache_path, "w") as f:
            json.dump(cache, f, indent=2)

    if newly_extracted:
        print(f"Extracted {newly_extracted} new case(s), cached to {cache_path}.")
    if failed_case_ids:
        print(f"WARNING: {len(failed_case_ids)} case(s) failed extraction and were "
              f"skipped: {failed_case_ids}. Re-run load_cases_from_file() again -- "
              f"already-cached cases won't be re-billed, only the failed ones will retry.")
    print(f"Loaded {len(cases)} case(s) total from {cases_path}"
          f"{' (' + str(len(failed_case_ids)) + ' skipped)' if failed_case_ids else ''}.")
    return cases


## 9. Scoring: matching predicted vs. ground-truth diagnoses

In [9]:
def _normalize_disorder(name: str) -> str:
    name = name.lower()
    name = re.sub(r"\bpersonality disorder\b", "", name)
    name = re.sub(r"[^a-z ]", "", name)
    return name.strip()


def _disorder_match(a: str, b: str) -> bool:
    na, nb = _normalize_disorder(a), _normalize_disorder(b)
    if not na or not nb:
        return False
    return na == nb or na in nb or nb in na


def _likelihood_weighted_match_details(predicted_sorted: list, match_pairs: list) -> list:
    """Give each correct diagnosis 0-100 credit relative to the report's top likelihood.
    A diagnosis absent from the report receives 0; the top prediction receives 100."""
    top_likelihood = max((float(item.get("likelihood_percent", 0) or 0) for item in predicted_sorted), default=0.0)
    details = []
    for ground_truth, matched_name in match_pairs:
        item = next((candidate for candidate in predicted_sorted
                     if matched_name and candidate.get("disorder") == matched_name), None)
        if item is None and matched_name:
            item = next((candidate for candidate in predicted_sorted
                         if _disorder_match(candidate.get("disorder", ""), matched_name)), None)
        likelihood = float(item.get("likelihood_percent", 0) or 0) if item else 0.0
        relative_percent = 100.0 * likelihood / top_likelihood if top_likelihood > 0 else 0.0
        details.append({"ground_truth": ground_truth,
                        "matched_predicted": item.get("disorder") if item else None,
                        "likelihood_percent": likelihood if item else None,
                        "relative_match_percent": round(relative_percent, 2)})
    return details


def score_report(predicted_assessment: list, ground_truth_diagnoses: list, k: int = None) -> dict:
    """Fast, free, substring-based matching. Brittle to abbreviations/rewordings
    (e.g. won't connect "OCPD" to "Obsessive-Compulsive Personality Disorder"
    unless one is literally a substring of the other) — prefer llm_score_report
    for anything you're actually reporting on. Kept as a zero-cost fallback."""
    k = k or len(predicted_assessment)
    predicted_sorted = sorted(
        predicted_assessment, key=lambda item: item.get("likelihood_percent", 0), reverse=True
    )
    top_k_items = predicted_sorted[:k]
    top_k_names = [item["disorder"] for item in top_k_items]
    match_pairs = []
    for gt in ground_truth_diagnoses:
        item = next((candidate for candidate in top_k_items if _disorder_match(gt, candidate["disorder"])), None)
        match_pairs.append((gt, item["disorder"] if item else None))
    matches = [(gt, matched_name is not None) for gt, matched_name in match_pairs]
    match_details = _likelihood_weighted_match_details(top_k_items, match_pairs)
    recall_at_k = (
        sum(1 for _, hit in matches if hit) / len(ground_truth_diagnoses)
        if ground_truth_diagnoses else None
    )
    true_positives = sum(1 for _, hit in matches if hit)
    precision_at_k = true_positives / len(top_k_names) if top_k_names else None
    f1_at_k = (2 * precision_at_k * recall_at_k / (precision_at_k + recall_at_k)
               if precision_at_k is not None and recall_at_k is not None and precision_at_k + recall_at_k else 0.0)
    likelihood_weighted_recall = (sum(item["relative_match_percent"] for item in match_details)
                                  / (100 * len(ground_truth_diagnoses)) if ground_truth_diagnoses else None)
    exact_set_match = bool(recall_at_k == 1.0 and len(top_k_names) == len(ground_truth_diagnoses)
                           and precision_at_k == 1.0)
    return {"ground_truth": ground_truth_diagnoses, "top_k_predicted": top_k_names,
            "matches": matches, "match_details": match_details, "recall_at_k": recall_at_k,
            "likelihood_weighted_recall": likelihood_weighted_recall, "precision_at_k": precision_at_k,
            "f1_at_k": f1_at_k, "exact_set_match": exact_set_match}


# ---- LLM-as-judge matching (recommended default) ----
#
# String matching mislabels a lot of genuinely correct predictions as misses (e.g.
# "Borderline Personality Disorder" vs "BPD", or a slightly different phrasing)
# and can't distinguish a real match from an unrelated disorder that happens to
# share a word. A small LLM judge call fixes both without a hand-written synonym
# list, at the cost of one extra call per scored case. The percentage math itself
# is unchanged (recall@k = matched / total, so 1 of 3 -> 33%) — only the matching
# decision gets smarter.

PERSONALITY_DISORDER_KEYWORDS = [
    "paranoid", "schizoid", "schizotypal", "antisocial", "borderline",
    "histrionic", "narcissistic", "avoidant", "dependent", "obsessive compulsive",
]


def is_personality_disorder(name: str) -> bool:
    """True if `name` is (or clearly refers to) one of the 10 DSM-5-TR personality
    disorders. THERAPIST_SYSTEM_PROMPT/REPORT_USER_TEMPLATE intentionally scope
    this system to ONLY those 10 — it will never predict a non-PD diagnosis (e.g.
    an anxiety or substance-use disorder) by design, even if a case's ground truth
    includes one as a comorbidity. Used to separate genuine PD misses from
    out-of-scope ground-truth items when reporting accuracy (see summarize_accuracy)."""
    n = _normalize_disorder(name)
    return any(kw in n for kw in PERSONALITY_DISORDER_KEYWORDS) or "personality" in name.lower()


DIAGNOSIS_JUDGE_SYSTEM_PROMPT = """You are a careful clinical rater comparing two lists of \
diagnoses for the SAME case: a GROUND-TRUTH list and a PREDICTED list from an independent \
assessment. For each ground-truth diagnosis, decide whether it is matched by any predicted \
diagnosis — meaning the predicted name refers to the SAME clinical entity (allow standard \
abbreviations and rewordings, e.g. "OCPD" = "Obsessive-Compulsive Personality Disorder", "BPD" \
= "Borderline Personality Disorder"). Do NOT count a merely related, comorbid, or similar-sounding \
condition as a match (e.g. Borderline Personality Disorder is NOT a match for Bipolar Disorder; \
Social Anxiety Disorder is NOT a match for Avoidant Personality Disorder, even though they often \
co-occur and share features).

Return ONLY valid JSON, no commentary, in exactly this shape:
{"matches": [{"ground_truth": "...", "matched_predicted": "exact string from PREDICTED list, or null if no match", "matched": true/false}]}
"""


def llm_score_report(predicted_assessment: list, ground_truth_diagnoses: list,
                      k: int = None, model: str = None) -> dict:
    """Same recall@k percentage as score_report, but the matching itself is judged
    by an LLM instead of substring comparison — correctly handles abbreviations,
    synonyms, and rewordings that substring matching misses."""
    if not ground_truth_diagnoses:
        return {"ground_truth": [], "top_k_predicted": [], "matches": [], "recall_at_k": None}

    k = k or len(predicted_assessment)
    predicted_sorted = sorted(
        predicted_assessment, key=lambda item: item.get("likelihood_percent", 0), reverse=True
    )
    top_k_items = predicted_sorted[:k]
    top_k_names = [item["disorder"] for item in top_k_items]

    user_prompt = (
        f"GROUND-TRUTH diagnoses:\n{json.dumps(ground_truth_diagnoses)}\n\n"
        f"PREDICTED diagnoses (top {k}, ranked by the assessor's own confidence):\n"
        f"{json.dumps(top_k_names)}"
    )
    text = chat(model or EVAL_JUDGE_MODEL, [
        {"role": "system", "content": DIAGNOSIS_JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ], temperature=0.0, max_tokens=800)
    parsed = _extract_json(text)

    match_pairs = [(m["ground_truth"], m.get("matched_predicted") if m.get("matched") else None)
                   for m in parsed["matches"]]
    matches = [(ground_truth, matched_name is not None) for ground_truth, matched_name in match_pairs]
    match_details = _likelihood_weighted_match_details(top_k_items, match_pairs)
    recall_at_k = (
        sum(1 for _, hit in matches if hit) / len(ground_truth_diagnoses)
        if ground_truth_diagnoses else None
    )
    true_positives = sum(1 for _, hit in matches if hit)
    precision_at_k = true_positives / len(top_k_names) if top_k_names else None
    f1_at_k = (2 * precision_at_k * recall_at_k / (precision_at_k + recall_at_k)
               if precision_at_k is not None and recall_at_k is not None and precision_at_k + recall_at_k else 0.0)
    likelihood_weighted_recall = (sum(item["relative_match_percent"] for item in match_details)
                                  / (100 * len(ground_truth_diagnoses)) if ground_truth_diagnoses else None)
    exact_set_match = bool(recall_at_k == 1.0 and len(top_k_names) == len(ground_truth_diagnoses)
                           and precision_at_k == 1.0)
    return {"ground_truth": ground_truth_diagnoses, "top_k_predicted": top_k_names,
            "matches": matches, "match_details": match_details, "recall_at_k": recall_at_k,
            "likelihood_weighted_recall": likelihood_weighted_recall, "precision_at_k": precision_at_k,
            "f1_at_k": f1_at_k, "exact_set_match": exact_set_match, "judge_raw": parsed}

## 10. Ensembled per-case evaluation

In [10]:
# ============================================================
# Ensembled diagnosis: run the SAME transcript through report generation
# n_repeats times (optionally across different models via report_models),
# then average. This separates:
#   - real ambiguity in what the Self agent expressed (a modeling finding)
#   - one-shot LLM sampling noise in the rater's judgment (measurement noise)
# ============================================================

def aggregate_assessments(assessment_runs: list) -> dict:
    likelihood_by_disorder, display_name_by_disorder = {}, {}
    for run in assessment_runs:
        for item in run:
            key = _normalize_disorder(item.get("disorder", ""))
            if not key:
                continue
            likelihood_by_disorder.setdefault(key, []).append(float(item.get("likelihood_percent", 0) or 0))
            display_name_by_disorder.setdefault(key, item.get("disorder"))

    averaged = []
    for key, likelihoods in likelihood_by_disorder.items():
        padded = likelihoods + [0.0] * (len(assessment_runs) - len(likelihoods))  # absent = 0 for that run
        averaged.append({
            "disorder": display_name_by_disorder[key],
            "likelihood_percent": sum(padded) / len(padded),
            "n_runs_included": len(likelihoods),
            "n_runs_total": len(assessment_runs),
        })
    averaged.sort(key=lambda d: d["likelihood_percent"], reverse=True)

    top1_per_run = [max(run, key=lambda i: i.get("likelihood_percent", 0))["disorder"] if run else None
                     for run in assessment_runs]
    top1_norm = [_normalize_disorder(d) for d in top1_per_run if d]
    agreement = (top1_norm.count(max(set(top1_norm), key=top1_norm.count)) / len(assessment_runs)
                 if top1_norm else 0.0)

    return {"assessment": averaged, "top1_agreement": agreement, "top1_per_run": top1_per_run}


def evaluate_case_ensembled(case: dict, min_turns: int = 6, max_turns: int = 8,
                             n_repeats: int = 3, n_disorders: int = 3,
                             report_models: list = None, verbose: bool = True) -> dict:
    load_case(case)
    initial_state: ConvState = {
        "history": [], "turn": 0, "min_turns": min_turns, "max_turns": max_turns,
        "decision": "", "last_trait_reactions": {}, "ocean_log": [], "report": "",
    }
    try:
        final_state = app.invoke(initial_state)
    except Exception as exc:
        return {"case_id": case["case_id"], "error": str(exc)}

    transcript = "\n".join(f'{t["speaker"].capitalize()}: {t["text"]}' for t in final_state["history"])
    models_cycle = report_models or [REPORT_MODEL]

    assessment_runs = []
    try:  # run #1 reuses the graph's own report call — no wasted API call
        assessment_runs.append(_extract_json(final_state["report"]).get("assessment", []))
    except Exception:
        pass
    for i in range(1, n_repeats):
        try:
            parsed = generate_report_assessment(transcript, n_disorders=n_disorders,
                                                  model=models_cycle[i % len(models_cycle)])
            assessment_runs.append(parsed["assessment"])
        except Exception as exc:
            if verbose:
                print(f'  (repeat {i+1}/{n_repeats} failed for {case["case_id"]} — {exc})')

    if not assessment_runs:
        return {"case_id": case["case_id"], "error": "all report repeats failed", "history": final_state["history"]}

    aggregated = aggregate_assessments(assessment_runs)
    ground_truth = case["eval_ground_truth"]["diagnoses"]
    try:
        score = llm_score_report(aggregated["assessment"], ground_truth)
    except Exception as exc:
        if verbose:
            print(f'  (LLM judge failed for {case["case_id"]} — {exc}; falling back to string matching)')
        score = score_report(aggregated["assessment"], ground_truth)

    result = {
        "case_id": case["case_id"], "turns": final_state["turn"], "history": final_state["history"],
        "ocean_log": final_state.get("ocean_log", []),
        "predicted": aggregated["assessment"], "top1_agreement": aggregated["top1_agreement"],
        "top1_per_run": aggregated["top1_per_run"], "n_repeats": len(assessment_runs), **score,
    }
    if verbose:
        recall_str = f'{result["recall_at_k"]:.0%}' if result["recall_at_k"] is not None else "n/a"
        print(f'\n=== {case["persona_name"]} ({case["case_id"]}) — ensembled x{len(assessment_runs)} ===')
        print("Ground truth:      ", result["ground_truth"])
        print("Top-k predicted:   ", result["top_k_predicted"])
        print("Recall@k:          ", recall_str)
        print("Top-1 agreement across repeats:", f'{result["top1_agreement"]:.0%}',
              "-> per-run top guess:", result["top1_per_run"])
    return result


def evaluate_cases_ensembled(cases: list, **kwargs) -> list:
    results = [evaluate_case_ensembled(case, **kwargs) for case in cases]
    valid = [r for r in results if "error" not in r and r["recall_at_k"] is not None]
    if valid:
        mean_recall = sum(r["recall_at_k"] for r in valid) / len(valid)
        mean_agreement = sum(r.get("top1_agreement", 0) for r in valid) / len(valid)
        print(f"\n{'='*60}\nEnsembled mean recall@k: {mean_recall:.1%} | "
              f"mean rater top-1 agreement: {mean_agreement:.1%}\n{'='*60}")
    return results

## 11. Aggregate accuracy: recall@k, R-Precision, MAP

In [11]:
def summarize_accuracy(results: list) -> dict:
    """Rolls per-case evaluate_cases_ensembled() results into one accuracy summary.

    This system is scoped to only assess the 10 DSM-5-TR PERSONALITY disorders (see
    THERAPIST_SYSTEM_PROMPT) — it can never predict a non-personality-disorder
    ground-truth item (e.g. an anxiety or substance-use disorder) by design. Those
    are reported separately as "out-of-scope" instead of being silently counted
    as ordinary misses, so the accuracy numbers reflect what the system was
    actually asked to do rather than penalizing it for something outside its brief.
    """
    valid = [r for r in results if "error" not in r and r["recall_at_k"] is not None]
    errors = [r for r in results if "error" in r]

    macro_recall = sum(r["recall_at_k"] for r in valid) / len(valid) if valid else None
    macro_likelihood_weighted_recall = (sum(r.get("likelihood_weighted_recall", 0.0) for r in valid)
                                       / len(valid) if valid else None)
    macro_precision = sum(r.get("precision_at_k", 0.0) for r in valid) / len(valid) if valid else None
    macro_f1 = sum(r.get("f1_at_k", 0.0) for r in valid) / len(valid) if valid else None
    exact_set_matches = sum(1 for r in valid if r.get("exact_set_match"))

    total_correct = sum(sum(1 for _, hit in r["matches"] if hit) for r in valid)
    total_ground_truth = sum(len(r["ground_truth"]) for r in valid)
    micro_recall = total_correct / total_ground_truth if total_ground_truth else None

    # In-scope-only breakdown: personality disorders only, excluding e.g. comorbid
    # Axis-I-style diagnoses (anxiety disorders, substance use disorders, ...) the
    # system was never asked to predict in the first place.
    in_scope_hits, out_of_scope_seen = [], []
    for r in valid:
        for gt, hit in r["matches"]:
            if is_personality_disorder(gt):
                in_scope_hits.append(hit)
            else:
                out_of_scope_seen.append(gt)
    in_scope_recall = (sum(in_scope_hits) / len(in_scope_hits)) if in_scope_hits else None

    fully_correct = sum(1 for r in valid if r["recall_at_k"] == 1.0)
    fully_missed = sum(1 for r in valid if r["recall_at_k"] == 0.0)

    summary = {
        "n_cases": len(results),
        "n_scored": len(valid),
        "n_errors": len(errors),
        "macro_recall_at_k": macro_recall,
        "final_rank_weighted_score": macro_likelihood_weighted_recall,
        "micro_recall_at_k": micro_recall,
        "macro_precision_at_k": macro_precision,
        "macro_f1_at_k": macro_f1,
        "exact_set_matches": exact_set_matches,
        "in_scope_recall_at_k": in_scope_recall,
        "out_of_scope_ground_truth_seen": sorted(set(out_of_scope_seen)),
        "cases_fully_correct": fully_correct,
        "cases_fully_missed": fully_missed,
    }
    in_scope_weighted = []
    for r in valid:
      case_weighted = [d["relative_match_percent"] for d in r.get("match_details", [])
            if is_personality_disorder(d["ground_truth"])]
      if case_weighted:
            in_scope_weighted.append(sum(case_weighted) / (100 * len(case_weighted)))
    in_scope_weighted_recall = sum(in_scope_weighted) / len(in_scope_weighted) if in_scope_weighted else None
    
    print("=" * 60)
    print("MODEL ACCURACY REPORT")
    print("=" * 60)
    print(f"Cases scored:            {summary['n_scored']}/{summary['n_cases']} "
          f"({summary['n_errors']} failed to parse a report)")
    print(f"Macro recall@k (mean of per-case recall): {macro_recall:.1%}"
          if macro_recall is not None else "Macro recall@k: n/a")
    print(f"OVERALL MODEL SCORE (rank-weighted recall): {macro_likelihood_weighted_recall:.1%}"
          if macro_likelihood_weighted_recall is not None else "OVERALL MODEL SCORE: n/a")
    print(f"Micro recall@k (pooled across all diagnoses): {micro_recall:.1%}"
          if micro_recall is not None else "Micro recall@k: n/a")
    print(f"Macro precision@k: {macro_precision:.1%}" if macro_precision is not None else "Macro precision@k: n/a")
    print(f"Macro F1@k:        {macro_f1:.1%}" if macro_f1 is not None else "Macro F1@k: n/a")
    print(f"Exact diagnosis sets: {exact_set_matches}/{len(valid)}")
    print(f"IN-SCOPE RANK-WEIGHTED RECALL (report this as THE accuracy number): "
          f"{in_scope_weighted_recall:.1%}" if in_scope_weighted_recall is not None else "n/a")
    if out_of_scope_seen:
        print(f"\nNOTE: {len(out_of_scope_seen)} ground-truth diagnosis mention(s) were NOT "
              f"personality disorders — this system only predicts the 10 DSM-5-TR PERSONALITY disorders, by design, "
              f"so it can never match these: {summary['out_of_scope_ground_truth_seen']}")
        print(f"In-scope-only recall@k (personality disorders only): {in_scope_recall:.1%}"
              if in_scope_recall is not None else "In-scope-only recall@k: n/a")
    print(f"\nCases fully correct (all ground-truth diagnoses found): {fully_correct}/{len(valid)}")
    print(f"Cases fully missed (none found):                        {fully_missed}/{len(valid)}")
    return summary


def _norm(s):
    return s.strip().lower()

def r_precision(ground_truth: list, predicted_names_ranked: list) -> float:
    R = len(ground_truth)
    if R == 0:
        return None
    top_r = [_norm(p) for p in predicted_names_ranked[:R]]
    return sum(1 for g in ground_truth if _norm(g) in top_r) / R

def average_precision(ground_truth: list, predicted_names_ranked: list) -> float:
    R = len(ground_truth)
    if R == 0:
        return None
    gt_norm = set(_norm(g) for g in ground_truth)
    pred_norm = [_norm(p) for p in predicted_names_ranked]
    hit_ranks = sorted(i + 1 for i, p in enumerate(pred_norm) if p in gt_norm)
    if not hit_ranks:
        return 0.0
    return sum((i + 1) / r for i, r in enumerate(hit_ranks)) / R


def summarize_accuracy_v2(results: list) -> dict:
    summary = summarize_accuracy(results)  # keeps all existing numbers/printing
    valid = [r for r in results if "error" not in r and r.get("recall_at_k") is not None]

    r_precisions = [r_precision(r["ground_truth"], r.get("top_k_predicted", [])) for r in valid]
    aps = [average_precision(r["ground_truth"], r.get("top_k_predicted", [])) for r in valid]
    r_precisions = [x for x in r_precisions if x is not None]
    aps = [x for x in aps if x is not None]

    summary["mean_r_precision"] = sum(r_precisions) / len(r_precisions) if r_precisions else None
    summary["map"] = sum(aps) / len(aps) if aps else None

    if summary["mean_r_precision"] is not None:
        print(f"\nMean R-Precision (k = #true diagnoses): {summary['mean_r_precision']:.1%}")
    if summary["map"] is not None:
        print(f"MAP (Mean Average Precision, rank-aware): {summary['map']:.1%}")
    return summary

## 12. Rater-validation baseline: textbook control vignettes

In [12]:
# ============================================================
# C. Rater-validation harness. Each vignette states TEXTBOOK, unambiguous
# features of ONE disorder, bypassing the Self agent/interview entirely.
# This gives you the rater's own ceiling — if it can't reliably recover an
# unambiguous case, a miss on a Self-agent case tells you more about the
# rater than about your modeling. Add more vignettes/disorders as you like.
# ============================================================

CONTROL_VIGNETTES = [
    {"case_id": "control_narcissistic", "persona_name": "Control-Narcissistic",
     "eval_ground_truth": {"diagnoses": ["Narcissistic personality disorder"]},
     "control_text": ("Self: Honestly, most people just can't keep up with me -- I'm smarter and more "
         "capable than nearly everyone I work with, and that's not arrogance, it's just true. I get "
         "furious when I don't get the credit I deserve. My last two relationships ended because my "
         "partners said I didn't care about their feelings -- I don't see the point of dwelling on "
         "other people's problems when I have my own goals. I only really respect people as "
         "successful as I am.")},
    {"case_id": "control_antisocial", "persona_name": "Control-Antisocial",
     "eval_ground_truth": {"diagnoses": ["Antisocial personality disorder"]},
     "control_text": ("Self: I've been arrested a few times, but most of it wasn't really my fault. I "
         "lie to people when it gets me what I want -- it's just smart. I've hit people when they made "
         "me angry and didn't feel bad about it after. I don't plan ahead; if something feels good "
         "right now I just do it. I forged some paperwork at a job, and when I got caught I just moved "
         "on and did the same thing somewhere else.")},
    {"case_id": "control_histrionic", "persona_name": "Control-Histrionic",
     "eval_ground_truth": {"diagnoses": ["Histrionic personality disorder"]},
     "control_text": ("Self: I feel awful if I'm not the center of attention in a room, like I "
         "physically can't relax until people notice me. I dress dramatically and my emotions swing "
         "fast, big tears one minute, laughing the next. I flirt with basically everyone, it's just how "
         "I get people to like me. My relationships move fast and I describe people as either amazing "
         "or the worst.")},
    {"case_id": "control_schizoid", "persona_name": "Control-Schizoid",
     "eval_ground_truth": {"diagnoses": ["Schizoid personality disorder"]},
     "control_text": ("Self: I genuinely don't want close relationships, and I know that sounds sad but "
         "it isn't to me. I'd rather spend a weekend alone than with anyone, even family. Compliments "
         "and criticism both roll off me the same way. I've never really wanted a romantic "
         "relationship. People say I seem cold or distant, but I don't experience much emotion to "
         "show.")},
    {"case_id": "control_paranoid", "persona_name": "Control-Paranoid",
     "eval_ground_truth": {"diagnoses": ["Paranoid personality disorder"]},
     "control_text": ("Self: I don't tell people much about myself because I know they'll use it "
         "against me eventually. I've cut off two friends because I was sure they were talking about "
         "me, even without real proof. I read into small comments -- if my boss reschedules a meeting "
         "I assume they're building a case against me. I still wonder if my partner is loyal, even "
         "now.")},
    {"case_id": "control_dependent", "persona_name": "Control-Dependent",
     "eval_ground_truth": {"diagnoses": ["Dependent personality disorder"]},
     "control_text": ("Self: I struggle to make even small decisions without checking with someone "
         "first -- what to order, what to wear, all of it. I stayed in a relationship way too long "
         "because I was terrified of being on my own. I go along with what others want even when I "
         "disagree, because I'm scared of upsetting them. I feel helpless when I'm alone too long.")},
]


def evaluate_control_vignettes(n_disorders: int = 3, verbose: bool = True) -> list:
    results = []
    for v in CONTROL_VIGNETTES:
        try:
            parsed = generate_report_assessment(v["control_text"], n_disorders=n_disorders)
        except Exception as exc:
            results.append({"case_id": v["case_id"], "error": str(exc)})
            continue
        ground_truth = v["eval_ground_truth"]["diagnoses"]
        try:
            score = llm_score_report(parsed["assessment"], ground_truth)
        except Exception:
            score = score_report(parsed["assessment"], ground_truth)
        result = {"case_id": v["case_id"],
                   "history": [{"speaker": "self", "text": v["control_text"]}],
                   "predicted": parsed["assessment"],
                   "overall_impression": parsed["overall_impression"], **score}
        if verbose:
            recall_str = f'{result["recall_at_k"]:.0%}' if result["recall_at_k"] is not None else "n/a"
            top1 = result["top_k_predicted"][0] if result["top_k_predicted"] else "n/a"
            print(f'{v["case_id"]:<24} true={ground_truth[0]:<40} top1={top1:<40} recall={recall_str}')
        results.append(result)
    return results

# Independent Therapist Baseline Evaluation
An independent Therapist baseline was evaluated on 10 real clinical cases to establish a reference point for the diagnostic performance of the proposed system. The Therapist received each case without the diagnosis section and independently assigned a likelihood from 0–100% to each of the ten DSM-5-TR personality disorders.


In [13]:
# ============================================================
# 12B. Independent Therapist validation on REAL clinical cases
# FIXED VERSION
#
# Fixes:
#   1. Case 8/9 preserve their out-of-scope diagnoses.
#   2. Literal JSON braces no longer break .format().
#   3. chat() is called with the correct argument order.
#   4. Ground truth is explicitly split into in-scope PDs and
#      out-of-scope diagnoses.
# ============================================================

import json
import os
import re
import math


DSM5TR_PERSONALITY_DISORDERS = [
    "Paranoid personality disorder",
    "Schizoid personality disorder",
    "Schizotypal personality disorder",
    "Antisocial personality disorder",
    "Borderline personality disorder",
    "Histrionic personality disorder",
    "Narcissistic personality disorder",
    "Avoidant personality disorder",
    "Dependent personality disorder",
    "Obsessive-compulsive personality disorder",
]


def _canonical_pd(name: str):
    """Map a diagnosis string to one of the 10 in-scope DSM-5-TR PD names."""
    if not name:
        return None

    n = _normalize_disorder(name)

    for pd in DSM5TR_PERSONALITY_DISORDERS:
        if _normalize_disorder(pd) == n:
            return pd

    aliases = {
        "ocpd": "Obsessive-compulsive personality disorder",
        "obsessive compulsive personality disorder":
            "Obsessive-compulsive personality disorder",
        "avoidant pd":
            "Avoidant personality disorder",
        "dependent pd":
            "Dependent personality disorder",
        "paranoid pd":
            "Paranoid personality disorder",
        "schizoid pd":
            "Schizoid personality disorder",
        "schizotypal pd":
            "Schizotypal personality disorder",
        "antisocial pd":
            "Antisocial personality disorder",
        "borderline pd":
            "Borderline personality disorder",
        "histrionic pd":
            "Histrionic personality disorder",
        "narcissistic pd":
            "Narcissistic personality disorder",
    }

    return aliases.get(n)


def _find_diagnosis_heading(lines):
    """
    Find the LAST standalone Diagnosis/Diagnoses heading.

    Searching from the end is safer because a vignette may mention
    the word 'diagnosis' earlier in the narrative.
    """
    for i in range(len(lines) - 1, -1, -1):
        normalized = lines[i].strip().lower()

        if normalized in {
            "diagnosis",
            "diagnoses",
            "diagnosis:",
            "diagnoses:",
        }:
            return i

    # Slightly more tolerant fallback.
    for i in range(len(lines) - 1, -1, -1):
        if re.fullmatch(r"\s*diagnos(?:is|es)\s*:?\s*", lines[i], re.I):
            return i

    return None


def extract_all_ground_truth(raw_text: str) -> list:
    """
    Extract every diagnosis explicitly listed in the terminal
    Diagnosis/Diagnoses section.

    Example:
        [
            "Avoidant personality disorder",
            "Social anxiety disorder"
        ]
    """
    lines = raw_text.strip("\n").split("\n")
    diag_idx = _find_diagnosis_heading(lines)

    if diag_idx is None:
        raise ValueError("No Diagnosis/Diagnoses section found in case text")

    diagnoses = []

    for line in lines[diag_idx + 1:]:
        text = line.strip()

        # Bullet format used by cases.json.
        text = re.sub(r"^[•\-\*]\s*", "", text).strip()

        if not text:
            continue

        diagnoses.append(text)

    if not diagnoses:
        raise ValueError(
            f"Diagnosis/Diagnoses heading found at line {diag_idx}, "
            "but no diagnoses were listed afterward."
        )

    return diagnoses


def split_ground_truth_diagnoses(all_diagnoses: list) -> dict:
    """
    Separate the complete case ground truth into:

      in_scope:
          the 10 DSM-5-TR personality disorders this system evaluates

      out_of_scope:
          other diagnoses explicitly stated in the case
          (e.g. Social Anxiety Disorder, Benzodiazepine Use Disorder)
    """
    in_scope = []
    out_of_scope = []

    for diagnosis in all_diagnoses:
        canonical = _canonical_pd(diagnosis)

        if canonical is not None:
            if canonical not in in_scope:
                in_scope.append(canonical)
        else:
            out_of_scope.append(diagnosis)

    return {
        "all": all_diagnoses,
        "in_scope": in_scope,
        "out_of_scope": out_of_scope,
    }


def extract_in_scope_ground_truth(raw_text: str) -> list:
    """
    Backward-compatible helper.

    Returns only the personality-disorder diagnoses because the
    Therapist baseline is explicitly scoped to the 10 PDs.
    """
    all_diagnoses = extract_all_ground_truth(raw_text)
    split = split_ground_truth_diagnoses(all_diagnoses)

    if not split["in_scope"]:
        raise ValueError(
            "No in-scope personality-disorder ground truth found. "
            f"All diagnoses found: {split['all']}"
        )

    return split["in_scope"]


def remove_diagnosis_section(raw_text: str) -> str:
    """
    Remove the terminal Diagnosis/Diagnoses section before sending
    the vignette to the independent Therapist.
    """
    lines = raw_text.strip("\n").split("\n")
    diag_idx = _find_diagnosis_heading(lines)

    if diag_idx is None:
        return raw_text.strip()

    return "\n".join(lines[:diag_idx]).strip()


# ------------------------------------------------------------
# Load and validate cases BEFORE making any model calls.
# ------------------------------------------------------------

with open(CASES_FILE, encoding="utf-8") as f:
    raw_cases = json.load(f)

cases = {
    str(case["case_id"]): case["raw_text"]
    for case in raw_cases
}

print(f"Validating {len(cases)} cases from {CASES_FILE}...\n")

validated_ground_truth = {}

for cid, text in cases.items():
    all_gt = extract_all_ground_truth(text)
    split_gt = split_ground_truth_diagnoses(all_gt)
    cleaned = remove_diagnosis_section(text)

    validated_ground_truth[cid] = split_gt

    print(
        f"Case {cid}: "
        f"ALL={split_gt['all']} | "
        f"IN-SCOPE={split_gt['in_scope']} | "
        f"OUT-OF-SCOPE={split_gt['out_of_scope']} | "
        f"held-out text={len(cleaned)} chars"
    )

print("\nAll cases passed validation.")


# ------------------------------------------------------------
# Therapist baseline prompts
# ------------------------------------------------------------

THERAPIST_BASELINE_SYSTEM_PROMPT = """
You are Dr. Reyes, an expert clinical psychologist conducting an independent
research evaluation of a clinical vignette.

Your task is NOT to simulate a conversation and NOT to use any diagnosis stated
in the vignette. The diagnosis section has been removed before you receive the
case. Based only on the clinical information provided, independently evaluate
ALL TEN DSM-5-TR personality disorders listed below.

Before scoring, write a brief TRIAGE in plain text: for each cluster (A, B, C),
name only the disorder(s) with real supporting evidence in the vignette; if a
cluster has none, say so in one clause. Then, for any disorder you are about to
score above 30, write 1-2 sentences citing the specific evidence and naming the
closest confusable disorder plus the one piece of evidence that separates them.
Do not write anything for disorders with no real evidence -- do not produce one
line per disorder, only for the ones actually in contention. This triage is
what keeps your numbers grounded instead of a vague overall impression.

For EACH of the ten disorders, assign a likelihood/confidence from 0 to 100
representing how strongly the presented case supports that the patient meets
the disorder's criteria. These are INDEPENDENT scores: they are NOT
probabilities competing for one outcome, they do NOT need to sum to 100, and
multiple disorders may receive high scores when the evidence supports
comorbidity.

CALIBRATION SCALE -- use this for every likelihood_percent, do not just guess a number:
- 0-10: No meaningful evidence, or only one isolated trait/episode.
- 11-30: A few relevant behaviors are present, but the pattern is not clearly
enduring or pervasive, is confined to one situation/relationship or a recent
stressor, or is better explained by a closely confusable disorder.
- 31-60: A partial pattern -- several required criteria have specific cited
evidence -- but pervasiveness across contexts, duration, or clinically
significant impairment is not fully established, OR a closely confusable
disorder is equally or more supported by the same evidence.
- 61-85: Most required criteria are met with specific evidence; the pattern
appears enduring and pervasive across contexts; the closest confusable
disorder is clearly less well supported.
- 86-100: Criteria are extensively and specifically met, pervasive across
multiple life domains, causing clear distress/impairment, and clearly
distinguished from every confusable alternative.

Use only evidence present in the vignette. Do not infer a disorder merely because
another diagnosis is mentioned or because a symptom is nonspecific. Distinguish
closely related disorders using the DSM-5-TR criteria and consider differential
diagnosis.

Return every one of the ten disorders exactly once, using the exact names
below, as the LAST thing in your response. For any disorder scoring 10 or
below, leave "rationale" as an empty string -- do not spend words justifying a
near-zero score. Return strict JSON and nothing after it.

DSM-5-TR personality disorders:
1. Paranoid personality disorder
2. Schizoid personality disorder
3. Schizotypal personality disorder
4. Antisocial personality disorder
5. Borderline personality disorder
6. Histrionic personality disorder
7. Narcissistic personality disorder
8. Avoidant personality disorder
9. Dependent personality disorder
10. Obsessive-compulsive personality disorder
""".strip()


# IMPORTANT:
# Literal JSON braces are doubled because this template is passed
# through .format(case_text=...).
THERAPIST_BASELINE_USER_TEMPLATE = """
Evaluate the following clinical case independently.

CASE:
{case_text}

First, write your brief TRIAGE as plain text, as instructed above -- only the
disorders actually in contention, not all ten.

Then return exactly this JSON structure with all ten disorders. Leave
"rationale" empty for any disorder scoring 10 or below:

{{
  "assessment": [
    {{"disorder": "Paranoid personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Schizoid personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Schizotypal personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Antisocial personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Borderline personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Histrionic personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Narcissistic personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Avoidant personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Dependent personality disorder", "likelihood_percent": 0, "rationale": ""}},
    {{"disorder": "Obsessive-compulsive personality disorder", "likelihood_percent": 0, "rationale": ""}}
  ]
}}
""".strip()


def _parse_therapist_baseline_response(text: str) -> dict:
    parsed = _extract_json(text)

    assessment = parsed.get("assessment")

    if not isinstance(assessment, list):
        raise ValueError(
            "Therapist baseline response has no assessment list"
        )

    by_pd = {}

    for item in assessment:
        if not isinstance(item, dict):
            continue

        pd = _canonical_pd(item.get("disorder", ""))

        if pd is None:
            continue

        value = float(item.get("likelihood_percent", 0))

        if not 0 <= value <= 100:
            raise ValueError(
                f"Invalid likelihood for {pd}: {value}"
            )

        # If a model accidentally repeats a disorder, keep the last value.
        by_pd[pd] = value

    missing = [
        pd
        for pd in DSM5TR_PERSONALITY_DISORDERS
        if pd not in by_pd
    ]

    if missing:
        raise ValueError(
            f"Therapist omitted DSM-5-TR disorders: {missing}"
        )

    return {
        "assessment": [
            {
                "disorder": pd,
                "likelihood_percent": by_pd[pd]
            }
            for pd in DSM5TR_PERSONALITY_DISORDERS
        ]
    }


def score_likelihoods(
    assessment: list,
    ground_truth: list
) -> dict:
    """
    Compute:
      - GTLS
      - multi-label Brier score
      - multi-label log loss
      - top-1 accuracy

    ground_truth MUST contain only in-scope personality disorders.
    """
    scores = {}

    for item in assessment:
        pd = _canonical_pd(item.get("disorder", ""))

        if pd is not None:
            scores[pd] = float(
                item.get("likelihood_percent", 0) or 0
            )

    gt = {
        _canonical_pd(x)
        for x in ground_truth
    }

    gt.discard(None)

    if not gt:
        raise ValueError(
            "No in-scope personality-disorder ground truth"
        )

    # IMPORTANT: the report is intentionally sparse. A disorder omitted from
    # the ranked list is treated as 0% evidence, not as an evaluation error.
    # This keeps sparsity compatible with the all-10-disorder scoring used by
    # Experiment 2 and prevents every case from being discarded.
    missing_scores = [
        pd
        for pd in DSM5TR_PERSONALITY_DISORDERS
        if pd not in scores
    ]
    for pd in missing_scores:
        scores[pd] = 0.0

    # --------------------------------------------------------
    # GTLS
    # --------------------------------------------------------

    gt_likelihoods = [
        scores.get(pd, 0.0)
        for pd in gt
    ]

    gtls = (
        sum(gt_likelihoods) / len(gt_likelihoods)
    )

    # --------------------------------------------------------
    # Multi-label Brier score
    # --------------------------------------------------------

    probs = [
        scores.get(pd, 0.0) / 100.0
        for pd in DSM5TR_PERSONALITY_DISORDERS
    ]

    labels = [
        1.0 if pd in gt else 0.0
        for pd in DSM5TR_PERSONALITY_DISORDERS
    ]

    brier = sum(
        (p - y) ** 2
        for p, y in zip(probs, labels)
    ) / len(labels)

    # --------------------------------------------------------
    # Multi-label log loss
    # --------------------------------------------------------

    eps = 1e-6

    log_loss = -sum(
        y * math.log(max(p, eps))
        + (1 - y) * math.log(max(1 - p, eps))
        for p, y in zip(probs, labels)
    ) / len(labels)

    # --------------------------------------------------------
    # Top-1
    # --------------------------------------------------------

    top1_pd = max(scores, key=scores.get)
    top1_correct = top1_pd in gt

    return {
        "ground_truth_likelihood_score": gtls,
        "ground_truth_likelihoods": {
            pd: scores.get(pd, 0.0)
            for pd in gt
        },
        "brier_score": brier,
        "log_loss": log_loss,
        "top1_prediction": top1_pd,
        "top1_correct": top1_correct,
    }


def evaluate_therapist_baseline(
    cases_path: str = CASES_FILE,
    verbose: bool = True
) -> list:
    """
    Run the Therapist alone on the real cases.

    The Therapist receives the vignette WITHOUT the diagnosis section.

    Complete ground truth is preserved in:
        ground_truth_all

    Personality-disorder ground truth used for scoring:
        ground_truth

    Out-of-scope diagnoses are preserved in:
        out_of_scope_ground_truth
    """

    with open(cases_path, encoding="utf-8") as f:
        cases_data = json.load(f)

    results = []

    # Keep checkpoint next to the cases file rather than relying on
    # the notebook's current working directory.
    checkpoint_path = os.path.join(
        os.path.dirname(os.path.abspath(cases_path)),
        "therapist_baseline_results.json"
    )

    existing = {}

    if os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, encoding="utf-8") as f:
                previous = json.load(f)

            existing = {
                str(r["case_id"]): r
                for r in previous
                if (
                    "error" not in r
                    and "case_id" in r
                )
            }

        except Exception:
            existing = {}

    print(
        f"\nLoaded {len(cases_data)} case(s) from "
        f"{cases_path} for independent Therapist validation."
    )

    for case in cases_data:

        case_id = str(case["case_id"])
        raw_text = case["raw_text"]

        # ----------------------------------------------------
        # Extract COMPLETE ground truth.
        # ----------------------------------------------------

        gt_info = split_ground_truth_diagnoses(
            extract_all_ground_truth(raw_text)
        )

        ground_truth_all = gt_info["all"]
        ground_truth = gt_info["in_scope"]
        out_of_scope_ground_truth = gt_info["out_of_scope"]

        # ----------------------------------------------------
        # Reuse successful checkpoint if available.
        # ----------------------------------------------------

        if case_id in existing:

            previous_result = existing[case_id]

            # Make sure older checkpoint results are not missing
            # the newly preserved ground-truth fields.
            previous_result["ground_truth_all"] = ground_truth_all
            previous_result["ground_truth"] = ground_truth
            previous_result["out_of_scope_ground_truth"] = (
                out_of_scope_ground_truth
            )

            results.append(previous_result)

            print(
                f"[checkpoint] Case {case_id}: "
                f"reused successful result | "
                f"GT={ground_truth}"
            )

            continue

        # ----------------------------------------------------
        # Hold out diagnosis section.
        # ----------------------------------------------------

        case_text = remove_diagnosis_section(raw_text)

        try:

            # IMPORTANT:
            # chat() is defined as:
            #     chat(model, messages, ...)
            #
            # Therefore model MUST be the first positional argument.
            raw = chat(
                THERAPIST_MODEL,
                [
                    {
                        "role": "system",
                        "content": THERAPIST_BASELINE_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": (
                            THERAPIST_BASELINE_USER_TEMPLATE.format(
                                case_text=case_text
                            )
                        )
                    }
                ],
                temperature=0.1,
                reasoning_effort="low",
                max_tokens=2800,
            )

            parsed = _parse_therapist_baseline_response(raw)

            scores = score_likelihoods(
                parsed["assessment"],
                ground_truth
            )

            result = {
                "case_id": case_id,

                # Complete source ground truth.
                "ground_truth_all": ground_truth_all,

                # Only the 10 PDs used for scoring.
                "ground_truth": ground_truth,

                # Explicitly preserved but excluded from PD scoring.
                "out_of_scope_ground_truth": (
                    out_of_scope_ground_truth
                ),

                "predicted": parsed["assessment"],

                **scores,
            }

        except Exception as exc:

            print(
                f"ERROR Case {case_id}: {type(exc).__name__}: {exc}"
            )

            result = {
                "case_id": case_id,
                "ground_truth_all": ground_truth_all,
                "ground_truth": ground_truth,
                "out_of_scope_ground_truth": (
                    out_of_scope_ground_truth
                ),
                "error": str(exc),
            }

        results.append(result)

        # ----------------------------------------------------
        # Checkpoint after EVERY case.
        # ----------------------------------------------------

        with open(
            checkpoint_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                results,
                f,
                indent=2,
                ensure_ascii=False
            )

        if "error" not in result and verbose:

            print(
                f"Case {case_id}: "
                f"GTLS={result['ground_truth_likelihood_score']:.1f}% | "
                f"Brier={result['brier_score']:.3f} | "
                f"Top-1={result['top1_prediction']} "
                f"({'✓' if result['top1_correct'] else '✗'})"
            )

            if out_of_scope_ground_truth:
                print(
                    f"  Out-of-scope diagnosis preserved: "
                    f"{out_of_scope_ground_truth}"
                )

    return results


def summarize_therapist_baseline(results: list) -> dict:

    valid = [
        r
        for r in results
        if "error" not in r
    ]

    if not valid:
        return {
            "n_cases": len(results),
            "n_scored": 0,
            "n_errors": len(results),
        }

    gtls = [
        r["ground_truth_likelihood_score"]
        for r in valid
    ]

    briers = [
        r["brier_score"]
        for r in valid
    ]

    log_losses = [
        r["log_loss"]
        for r in valid
    ]

    top1_acc = (
        sum(r["top1_correct"] for r in valid)
        / len(valid)
    )

    per_disorder = {}

    for pd in DSM5TR_PERSONALITY_DISORDERS:

        vals = []

        for r in valid:

            item = next(
                (
                    x
                    for x in r["predicted"]
                    if x["disorder"] == pd
                ),
                None
            )

            if item is not None:
                vals.append(
                    float(item["likelihood_percent"])
                )

        per_disorder[pd] = (
            sum(vals) / len(vals)
            if vals
            else None
        )

    # Collect out-of-scope diagnoses that were present
    # in the source cases.
    out_of_scope_seen = sorted({
        diagnosis
        for r in valid
        for diagnosis in r.get(
            "out_of_scope_ground_truth",
            []
        )
    })

    summary = {
        "n_cases": len(results),
        "n_scored": len(valid),
        "n_errors": len(results) - len(valid),

        "mean_ground_truth_likelihood_score":
            sum(gtls) / len(gtls),

        "mean_brier_score":
            sum(briers) / len(briers),

        "mean_log_loss":
            sum(log_losses) / len(log_losses),

        "top1_accuracy":
            top1_acc,

        "per_disorder_mean_likelihood":
            per_disorder,

        # Transparency: diagnoses present in source cases
        # but outside the 10-PD evaluation scope.
        "out_of_scope_ground_truth_seen":
            out_of_scope_seen,
    }

    print("\n" + "=" * 70)
    print(
        "INDEPENDENT THERAPIST VALIDATION — "
        "REAL CLINICAL CASES"
    )
    print("=" * 70)

    print(
        f"Cases scored:                    "
        f"{len(valid)}/{len(results)}"
    )

    print(
        f"MEAN GROUND-TRUTH LIKELIHOOD:    "
        f"{summary['mean_ground_truth_likelihood_score']:.1%}"
    )

    print(
        f"Mean multi-label Brier score:     "
        f"{summary['mean_brier_score']:.3f} "
        f"(lower is better)"
    )

    print(
        f"Mean multi-label log loss:        "
        f"{summary['mean_log_loss']:.3f} "
        f"(lower is better)"
    )

    print(
        f"Top-1 accuracy (secondary):       "
        f"{summary['top1_accuracy']:.1%}"
    )

    if out_of_scope_seen:
        print(
            "\nOut-of-scope diagnoses preserved "
            "but excluded from PD scoring:"
        )

        for diagnosis in out_of_scope_seen:
            print(f"  - {diagnosis}")

    print(
        "\nThis baseline is the Therapist's own "
        "recognition ceiling on real cases."
    )

    print(
        "For simulation fidelity, compare the "
        "Self-agent pipeline against this baseline."
    )

    return summary


# ============================================================
# RUN
# ============================================================

therapist_baseline_results = evaluate_therapist_baseline(
    cases_path=CASES_FILE
)

therapist_baseline_summary = summarize_therapist_baseline(
    therapist_baseline_results
)

Validating 10 cases from cases.json...

Case 1: ALL=['Paranoid personality disorder', 'Obsessive-compulsive personality disorder'] | IN-SCOPE=['Paranoid personality disorder', 'Obsessive-compulsive personality disorder'] | OUT-OF-SCOPE=[] | held-out text=3523 chars
Case 2: ALL=['Schizoid personality disorder'] | IN-SCOPE=['Schizoid personality disorder'] | OUT-OF-SCOPE=[] | held-out text=3624 chars
Case 3: ALL=['Schizotypal personality disorder', 'Paranoid personality disorder'] | IN-SCOPE=['Schizotypal personality disorder', 'Paranoid personality disorder'] | OUT-OF-SCOPE=[] | held-out text=3200 chars
Case 4: ALL=['Antisocial personality disorder'] | IN-SCOPE=['Antisocial personality disorder'] | OUT-OF-SCOPE=[] | held-out text=4410 chars
Case 5: ALL=['Borderline personality disorder'] | IN-SCOPE=['Borderline personality disorder'] | OUT-OF-SCOPE=[] | held-out text=3137 chars
Case 6: ALL=['Histrionic personality disorder'] | IN-SCOPE=['Histrionic personality disorder'] | OUT-OF-SCOPE=

## 13. Run the full study, averaged over independent rounds

`run_full_study` runs the rater baseline plus every modeled case once. `run_full_study_multi_round` repeats that `n_rounds` times — each round is a brand-new set of therapist↔self dialogues, nothing is cached or reused — and reports mean ± stdev across rounds. That average is the number to put in a report.

In [14]:
# ============================================================
# D. Run both halves and report modeled recall RELATIVE to rater baseline —
# this is the number that actually supports "the multi-agent system models
# personality disorders in a way a clinician-style rater can recognize."
#
# TIP on model choice: pass report_models=[...] with 2+ strong models (e.g.
# ["openai/gpt-4.1", "anthropic/claude-sonnet-4.5"]) to also rule out
# single-model idiosyncrasy in the rater, not just sampling noise.
# ============================================================

def run_full_study(cases_path: str = CASES_FILE, n_repeats: int = 3, report_models: list = None,
                    output_path: str = "full_study_results.json") -> dict:
    print("### Step 1/2 — Rater baseline on textbook control vignettes ###\n")
    control_results = evaluate_control_vignettes()
    control_summary = summarize_accuracy_v2(control_results)

    print(f"\n\n### Step 2/2 — Self-agent-modeled cases (ensembled report, x{n_repeats} repeats) ###\n")
    all_cases = load_cases_from_file(cases_path)
    modeled_results = evaluate_cases_ensembled(all_cases, n_repeats=n_repeats, report_models=report_models)
    modeled_summary = summarize_accuracy_v2(modeled_results)

    baseline_recall = control_summary.get("in_scope_recall_at_k") or control_summary.get("macro_recall_at_k")
    modeled_recall = modeled_summary.get("in_scope_recall_at_k") or modeled_summary.get("macro_recall_at_k")

    # Guard against EITHER side being None (not just baseline_recall): modeled_recall
    # is None whenever every modeled case ended up in the "errors" bucket (e.g. the
    # report-generation JSON failed to parse for all of them), which previously
    # crashed here with `NoneType / float`.
    if baseline_recall is None or modeled_recall is None or baseline_recall == 0:
        relative_recognizability = None
        if modeled_recall is None:
            print(f"\nNOTE: modeled_recall is n/a — {modeled_summary.get('n_errors')}/"
                  f"{modeled_summary.get('n_cases')} modeled case(s) failed to score "
                  f"(see 'n_errors' above; likely a report-JSON parsing failure).")
        if baseline_recall is None:
            print(f"\nNOTE: baseline_recall is n/a — {control_summary.get('n_errors')}/"
                  f"{control_summary.get('n_cases')} control vignette(s) failed to score.")
    else:
        relative_recognizability = modeled_recall / baseline_recall

    print("\n" + "=" * 60)
    print("RECOGNIZABILITY RELATIVE TO RATER BASELINE")
    print("=" * 60)
    if baseline_recall is not None:
        print(f"Rater baseline recall (textbook vignettes): {baseline_recall:.1%}")
    if modeled_recall is not None:
        print(f"Self-agent-modeled recall (ensembled):       {modeled_recall:.1%}")
    if relative_recognizability is not None:
        print(f"Relative recognizability (modeled / baseline): {relative_recognizability:.1%}")
        print("-> near 100% = Self-agent personas are about as recognizable as textbook cases;")
        print("   well below 100% = a real modeling gap, not rater noise.")

    with open(output_path, "w") as f:
        json.dump({"control_summary": control_summary, "control_results": control_results,
                    "modeled_summary": modeled_summary, "modeled_results": modeled_results,
                    "relative_recognizability": relative_recognizability}, f, indent=2)
    print(f"\nFull results written to {output_path}")

    return {"control_summary": control_summary, "modeled_summary": modeled_summary,
            "relative_recognizability": relative_recognizability}


# For a single round, call run_full_study(n_repeats=3) directly.
# For a full report, run the multi-round version in the next cell instead (5
# independent rounds -- each with a brand-new therapist<->self dialogue -- averaged.

In [15]:
import statistics

def _mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None

def _stdev(values):
    values = [v for v in values if v is not None]
    return statistics.pstdev(values) if len(values) > 1 else 0.0 if values else None


def run_full_study_multi_round(cases_path: str = CASES_FILE, n_rounds: int = 5, n_repeats: int = 3,
                                 report_models: list = None,
                                 output_path: str = "full_study_results_multi_round.json") -> dict:
    """Runs run_full_study n_rounds times -- each round is a completely fresh
    call, so every round gets brand-new therapist<->self dialogues for every
    case (load_cases_from_file only caches the extracted case DEFINITIONS,
    never a session or its result, so nothing here is stale/reused) -- and
    reports the mean and population stdev of the headline numbers across
    rounds. Each round's full per-case detail (including every round's
    ocean_log) is kept in "rounds" so nothing is thrown away; "averaged" is
    the number to put in a report/slide.
    """
    round_outputs = []
    for round_idx in range(1, n_rounds + 1):
        print("\n" + "#" * 70)
        print(f"# ROUND {round_idx}/{n_rounds}")
        print("#" * 70)
        round_path = output_path.replace(".json", f"_round{round_idx}.json")
        result = run_full_study(cases_path=cases_path, n_repeats=n_repeats,
                                  report_models=report_models, output_path=round_path)
        # run_full_study() only returns the compact summary dict; reload the file
        # it just wrote so "rounds" below keeps full per-case detail too (transcripts,
        # ocean_log, per-case scores) rather than just the three headline numbers.
        with open(round_path) as f:
            full_round = json.load(f)
        round_outputs.append(full_round)

    def get_control(r, key):
        return r["control_summary"].get(key)

    def get_modeled(r, key):
        return r["modeled_summary"].get(key)

    metrics = {
        "control_in_scope_recall_at_k": [get_control(r, "in_scope_recall_at_k") for r in round_outputs],
        "control_macro_recall_at_k": [get_control(r, "macro_recall_at_k") for r in round_outputs],
        "control_mean_r_precision": [get_control(r, "mean_r_precision") for r in round_outputs],
        "control_map": [get_control(r, "map") for r in round_outputs],
        "modeled_in_scope_recall_at_k": [get_modeled(r, "in_scope_recall_at_k") for r in round_outputs],
        "modeled_macro_recall_at_k": [get_modeled(r, "macro_recall_at_k") for r in round_outputs],
        "modeled_mean_r_precision": [get_modeled(r, "mean_r_precision") for r in round_outputs],
        "modeled_map": [get_modeled(r, "map") for r in round_outputs],
        "relative_recognizability": [r.get("relative_recognizability") for r in round_outputs],
    }
    averaged = {
        name: {"mean": _mean(values), "stdev": _stdev(values), "per_round": values}
        for name, values in metrics.items()
    }

    print("\n\n" + "=" * 70)
    print(f"AVERAGED OVER {n_rounds} ROUNDS (mean ± population stdev)")
    print("=" * 70)
    def _fmt(stat):
        if stat["mean"] is None:
            return "n/a"
        return f'{stat["mean"]:.1%} ± {stat["stdev"]:.1%}' if stat["stdev"] is not None else f'{stat["mean"]:.1%}'

    print("Control (rater baseline):")
    print(f"  In-scope recall@k:   {_fmt(averaged['control_in_scope_recall_at_k'])}")
    print(f"  Macro recall@k:      {_fmt(averaged['control_macro_recall_at_k'])}")
    print(f"  R-Precision:         {_fmt(averaged['control_mean_r_precision'])}")
    print(f"  MAP:                 {_fmt(averaged['control_map'])}")
    print("Modeled (Self-agent-simulated cases):")
    print(f"  In-scope recall@k:   {_fmt(averaged['modeled_in_scope_recall_at_k'])}")
    print(f"  Macro recall@k:      {_fmt(averaged['modeled_macro_recall_at_k'])}")
    print(f"  R-Precision:         {_fmt(averaged['modeled_mean_r_precision'])}")
    print(f"  MAP:                 {_fmt(averaged['modeled_map'])}")
    print(f"Relative recognizability: {_fmt(averaged['relative_recognizability'])}")

    with open(output_path, "w") as f:
        json.dump({"n_rounds": n_rounds, "rounds": round_outputs, "averaged": averaged}, f, indent=2)
    print(f"\nAll {n_rounds} rounds (full per-case detail, incl. ocean_log) plus the averaged "
          f"summary were written to {output_path}")
    print(f"Each individual round was also written separately to "
          f"{output_path.replace('.json', '_round<N>.json')}")

    return {"rounds": round_outputs, "averaged": averaged}


# Run it: 5 independent rounds, each a fresh set of therapist<->self dialogues.
# study_multi_round = run_full_study_multi_round(n_rounds=5, n_repeats=3)
# Optional: run_full_study(n_repeats=3)  # DO NOT auto-run; Experiment 2 below is the modeled run.

# Evaluation Experiments
Experiment 1 — Therapist validation

Real clinical case → Therapist → diagnosis scores

In [16]:
# ============================================================
# EXPERIMENT 1 — additional metrics: confusion matrix, macro-F1
# ============================================================

def confusion_matrix(results):
    """Rows = primary true disorder, cols = top-1 predicted."""
    valid = [r for r in results if "error" not in r]
    m = {t: {p: 0 for p in DSM5TR_PERSONALITY_DISORDERS} for t in DSM5TR_PERSONALITY_DISORDERS}
    for r in valid:
        true_pd = r["ground_truth"][0]
        m[true_pd][r["top1_prediction"]] += 1
    return m

def print_confusion_matrix(matrix):
    labels = DSM5TR_PERSONALITY_DISORDERS
    short = {l: l.replace(" personality disorder", "")[:10] for l in labels}
    print("true\\pred".ljust(12) + "".join(short[l].rjust(7) for l in labels))
    for t in labels:
        print(short[t].ljust(12) + "".join(str(matrix[t][p]).rjust(7) for p in labels))

def macro_f1_from_threshold(results, threshold=50.0):
    """Binarize each disorder's likelihood at `threshold`; per-disorder P/R/F1,
    macro-F1, and Hamming accuracy across all case×disorder pairs."""
    valid = [r for r in results if "error" not in r]
    per_disorder_f1, correct, total = {}, 0, 0
    for pd in DSM5TR_PERSONALITY_DISORDERS:
        tp = fp = fn = tn = 0
        for r in valid:
            item = next((x for x in r["predicted"] if _canonical_pd(x.get("disorder","")) == pd), None)
            score = float(item.get("likelihood_percent", 0)) if item else 0.0
            pred_pos = score >= threshold
            true_pos = pd in r["ground_truth"]
            if pred_pos and true_pos: tp += 1
            elif pred_pos and not true_pos: fp += 1
            elif not pred_pos and true_pos: fn += 1
            else: tn += 1
            total += 1
            correct += (pred_pos == true_pos)
        precision = tp/(tp+fp) if (tp+fp) else 0.0
        recall = tp/(tp+fn) if (tp+fn) else 0.0
        f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
        per_disorder_f1[pd] = {"precision": precision, "recall": recall, "f1": f1}
    macro_f1 = sum(v["f1"] for v in per_disorder_f1.values()) / len(per_disorder_f1)
    return {"per_disorder_f1": per_disorder_f1, "macro_f1": macro_f1,
            "hamming_accuracy": correct/total if total else None}

def per_disorder_summary(results):
    valid = [r for r in results if "error" not in r]
    out = {}
    for pd in DSM5TR_PERSONALITY_DISORDERS:
        cases_with_pd = [r for r in valid if pd in r["ground_truth"]]
        if not cases_with_pd:
            out[pd] = None
            continue
        mean_lik = sum(r["ground_truth_likelihoods"].get(pd, 0.0) for r in cases_with_pd) / len(cases_with_pd)
        recog = sum(1 for r in cases_with_pd if r["top1_prediction"] == pd) / len(cases_with_pd)
        out[pd] = {"n_cases": len(cases_with_pd), "mean_target_likelihood": mean_lik, "recognition_rate": recog}
    return out

# run on the Experiment 1 results you already have
exp1_cm = confusion_matrix(therapist_baseline_results)
print_confusion_matrix(exp1_cm)
exp1_f1 = macro_f1_from_threshold(therapist_baseline_results)
_ham = exp1_f1['hamming_accuracy']
print(f"\nAccuracy (Hamming): {_ham:.1%}" if _ham is not None else "\nAccuracy (Hamming): n/a (no valid results)")
print(f"Macro-F1: {exp1_f1['macro_f1']:.1%}")
for pd, v in exp1_f1["per_disorder_f1"].items():
    print(f"  {pd:<45} F1={v['f1']:.2f}")
exp1_per_disorder = per_disorder_summary(therapist_baseline_results)

true\pred   ParanoidSchizoidSchizotypaAntisocialBorderlineHistrionicNarcissistAvoidantDependentObsessive-
Paranoid          1      0      0      0      0      0      0      0      0      0
Schizoid          0      0      1      0      0      0      0      0      0      0
Schizotypa        0      0      1      0      0      0      0      0      0      0
Antisocial        0      0      0      1      0      0      0      0      0      0
Borderline        0      0      0      0      1      0      0      0      0      0
Histrionic        0      0      0      0      0      1      0      0      0      0
Narcissist        0      0      0      0      0      0      1      0      0      0
Avoidant          0      0      0      0      0      0      0      1      0      0
Dependent         0      0      0      0      0      0      0      0      1      0
Obsessive-        0      0      0      0      0      0      0      0      0      1

Accuracy (Hamming): 97.0%
Macro-F1: 90.0%
  Paranoid personalit

Experiment 2 — Self-agent simulation

Target disorder → Self agent → Therapist dialogue → Therapist report
It reports:

Target diagnosis recognition rate
Mean target-diagnosis likelihood
Brier score
Log loss
Per-disorder target likelihood

In [ ]:
# ============================================================
# EXPERIMENT 2 — Self-agent simulation, scored the SAME way as
# Experiment 1 (n_disorders=10, all 10 always scored) so it's
# directly comparable for Experiment 3.
# NOTE: sparse reports may omit disorders; the scorer treats omissions as 0%.
# ============================================================

def evaluate_self_agent_cases(cases_path=CASES_FILE, n_repeats=3, n_disorders=10,
                             verbose=True, checkpoint_path="experiment2_checkpoint.json"):
    """Run modeled cases with visible progress and per-case checkpoints.

    The checkpoint makes long API experiments resumable: a completed case is
    written immediately, so a later provider timeout/interrupt does not erase
    the useful results already obtained.
    """
    all_cases = load_cases_from_file(cases_path)
    results = []

    # Resume only from a checkpoint that belongs to the same case file.
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path) as f:
                cached = json.load(f)
            cached_results = cached.get("results", [])
            cached_ids = {str(r.get("case_id")) for r in cached_results}
            if cached.get("cases_path") == cases_path:
                results.extend(cached_results)
                print(f"Resuming Experiment 2: {len(cached_ids)}/{len(all_cases)} case(s) already completed.")
        except Exception as exc:
            print(f"Checkpoint ignored: {exc}")

    done_ids = {str(r.get("case_id")) for r in results}
    total = len(all_cases)

    for idx, case in enumerate(all_cases, 1):
        case_id = str(case["case_id"])
        if case_id in done_ids:
            continue

        print(f"\n--- Experiment 2: case {idx}/{total} — {case.get('persona_name', case_id)} ---")
        started = time.time()
        try:
            r = evaluate_case_ensembled(
                case, n_repeats=n_repeats, n_disorders=n_disorders, verbose=True
            )
            if "error" in r:
                result = r
            else:
                gt = validated_ground_truth[case_id]["in_scope"]
                try:
                    scores = score_likelihoods(r["predicted"], gt)
                    result = {"case_id": case_id, "ground_truth": gt,
                              "predicted": r["predicted"], "history": r["history"],
                              **scores}
                    if verbose:
                        print(f"Case {case_id}: GTLS={result['ground_truth_likelihood_score']:.1f}% | "
                              f"Brier={result['brier_score']:.3f} | Top-1={result['top1_prediction']} "
                              f"({'✓' if result['top1_correct'] else '✗'})")
                except Exception as exc:
                    result = {"case_id": case_id, "error": f"scoring failed: {exc}"}
        except KeyboardInterrupt:
            print("\nInterrupted. Completed cases have been checkpointed; rerun this cell to resume.")
            raise
        except Exception as exc:
            result = {"case_id": case_id, "error": f"case execution failed: {exc}"}

        results.append(result)
        done_ids.add(case_id)
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump({"cases_path": cases_path, "results": results}, f, indent=2)
        print(f"Finished case {idx}/{total} in {(time.time()-started)/60:.1f} min.")

    return results

self_agent_results = evaluate_self_agent_cases()
exp2_cm = confusion_matrix(self_agent_results)
print_confusion_matrix(exp2_cm)
exp2_f1 = macro_f1_from_threshold(self_agent_results)
_n_errors = sum(1 for r in self_agent_results if "error" in r)
if _n_errors:
    print(f"\n{_n_errors}/{len(self_agent_results)} case(s) errored — see 'ERROR' lines above for details.")
_ham2 = exp2_f1['hamming_accuracy']
if _ham2 is not None:
    print(f"Accuracy (Hamming): {_ham2:.1%} | Macro-F1: {exp2_f1['macro_f1']:.1%}")
else:
    print("Accuracy (Hamming): n/a (no valid results) | Macro-F1: n/a")
exp2_per_disorder = per_disorder_summary(self_agent_results)

### 13b. Cheap 5-round average for Experiment 2

`run_full_study_multi_round` (section 13) is the most expensive way to get a multi-round
average: every round it also redoes the 6 control-vignette baseline calls, which don't need
to change round to round since Experiment 1 already has that baseline on file.

`run_experiment2_multi_round` below reruns *only* the self-agent simulation, `n_rounds` times,
each with a brand-new set of therapist↔self dialogues (nothing cached/reused, same guarantee as
`run_full_study_multi_round`), and reports mean ± stdev of the headline metrics across rounds.
That average — not any single round, including the one already saved in
`self_agent_results` above — is the number to put in a report.

Cost levers if you still want to trim further:
- `n_repeats` here only controls report-generation resampling on a *fixed* transcript — it's
  cheap. The dialogue itself (interpret + 5 trait agents + consensus, per turn) is the expensive
  part and runs exactly once per case per round regardless of `n_repeats`.
- Don't pass multiple `report_models` unless you specifically want to test rater-model
  idiosyncrasy — each extra model multiplies report-generation calls.
- Lowering `max_turns`/`min_turns` in `evaluate_case_ensembled` directly cuts the number of
  dialogue turns, which is the single biggest cost driver.

This cell only *defines* the function — it does not call it automatically, since a 5-round run
still means 5× the full 10-case dialogue cost. Run the cell below it explicitly when you're ready
to spend that budget.

In [24]:
# ============================================================
# EXPERIMENT 2b — cheap multi-round average (self-agent only,
# reuses the Experiment 1 control baseline instead of
# recomputing it every round).
# ============================================================

def run_experiment2_multi_round(n_rounds: int = 5, n_repeats: int = 1,
                                  n_disorders: int = 10, cases_path: str = CASES_FILE,
                                  checkpoint_prefix: str = "exp2_round",
                                  output_path: str = "experiment2_multi_round_results.json") -> dict:
    """Reruns evaluate_self_agent_cases n_rounds times -- a fresh checkpoint_path
    each round means nothing is resumed/skipped from a previous run, so every round
    gets brand-new therapist<->self dialogues for every case, same guarantee as
    run_full_study_multi_round. Returns per-round results plus mean +/- population
    stdev of the headline metrics across rounds -- report the average, not any
    single round.
    """
    round_results = []
    for round_idx in range(1, n_rounds + 1):
        print("\n" + "#" * 70)
        print(f"# EXPERIMENT 2 -- ROUND {round_idx}/{n_rounds}")
        print("#" * 70)
        results = evaluate_self_agent_cases(
            cases_path=cases_path,
            n_repeats=n_repeats,
            n_disorders=n_disorders,
            checkpoint_path=f"{checkpoint_prefix}{round_idx}.json",
        )
        round_results.append(results)

    def _round_means(results, key):
        valid = [r[key] for r in results if key in r and r[key] is not None]
        return sum(valid) / len(valid) if valid else None

    metrics = {}
    for key in ["ground_truth_likelihood_score", "brier_score", "log_loss"]:
        per_round = [_round_means(r, key) for r in round_results]
        per_round = [v for v in per_round if v is not None]
        metrics[key] = {
            "per_round": per_round,
            "mean": statistics.mean(per_round) if per_round else None,
            "stdev": statistics.pstdev(per_round) if len(per_round) > 1 else 0.0,
        }

    top1_rates = []
    for r in round_results:
        valid = [x for x in r if "top1_correct" in x]
        if valid:
            top1_rates.append(sum(1 for x in valid if x["top1_correct"]) / len(valid))
    metrics["top1_accuracy"] = {
        "per_round": top1_rates,
        "mean": statistics.mean(top1_rates) if top1_rates else None,
        "stdev": statistics.pstdev(top1_rates) if len(top1_rates) > 1 else 0.0,
    }

    print("\n" + "=" * 60)
    print(f"EXPERIMENT 2 -- {n_rounds}-ROUND AVERAGE (report this number)")
    print("=" * 60)
    for key, m in metrics.items():
        if m["mean"] is None:
            print(f"{key}: n/a")
        else:
            print(f"{key}: {m['mean']:.3f} +/- {m['stdev']:.3f}  (per round: {[round(v,3) for v in m['per_round']]})")

    output = {"n_rounds": n_rounds, "n_repeats": n_repeats, "metrics": metrics,
               "round_results": round_results}
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\nFull per-round results written to {output_path}")
    return output


# Uncomment to actually run -- this is still 5x the full 10-case dialogue cost:
experiment2_multi_round_results = run_experiment2_multi_round(n_rounds=5, n_repeats=1)


######################################################################
# EXPERIMENT 2 -- ROUND 1/5
######################################################################
Loaded 10 case(s) total from cases.json.
Resuming Experiment 2: 10/10 case(s) already completed.

######################################################################
# EXPERIMENT 2 -- ROUND 2/5
######################################################################
Loaded 10 case(s) total from cases.json.
Resuming Experiment 2: 10/10 case(s) already completed.

######################################################################
# EXPERIMENT 2 -- ROUND 3/5
######################################################################
Loaded 10 case(s) total from cases.json.
Resuming Experiment 2: 2/10 case(s) already completed.

--- Experiment 2: case 3/10 — Henry ---
Loaded persona "Henry" (3) — 6 memories

=== Henry (3) — ensembled x1 ===
Ground truth:       ['Schizotypal personality disorder', 'Paranoid personality dis

In [35]:
import json

# Use round 1 as the canonical single-round Experiment 2 result
# (already complete, 10/10 real cases, never touched by anything since)
with open("exp2_round1.json") as f:
    round1 = json.load(f)

self_agent_results = round1["results"]

with open("experiment2_checkpoint.json", "w") as f:
    json.dump({"results": self_agent_results}, f, indent=2)

print(f"self_agent_results set: {len(self_agent_results)} case(s)")
print("experiment2_checkpoint.json written")

self_agent_results set: 10 case(s)
experiment2_checkpoint.json written


Experiment 3 — Relative Simulation Fidelity

It compares Experiment 2 against Experiment 1:

Relative Simulation Fidelity=
Mean target likelihood in real cases
Mean target likelihood in simulations
	​


In [36]:
# ============================================================
# EXPERIMENT 3 — Relative Simulation Fidelity
# RSF_d = simulation recognition_d / therapist-baseline recognition_d
# ============================================================

def compute_rsf(exp2_results, exp1_results):
    exp2_pd = per_disorder_summary(exp2_results)
    exp1_pd = per_disorder_summary(exp1_results)
    rsf = {}
    for pd in DSM5TR_PERSONALITY_DISORDERS:
        sim, base = exp2_pd.get(pd), exp1_pd.get(pd)
        rsf[pd] = None if not sim or not base or base["recognition_rate"] == 0 \
            else sim["recognition_rate"] / base["recognition_rate"]
    valid = [v for v in rsf.values() if v is not None]
    overall = sum(valid) / len(valid) if valid else None
    print("="*60); print("EXPERIMENT 3 — RELATIVE SIMULATION FIDELITY"); print("="*60)
    for pd in DSM5TR_PERSONALITY_DISORDERS:
        v = rsf[pd]
        print(f"{pd:<45} {'n/a' if v is None else f'{v:.2f}'}")
    print(f"\nOverall RSF: {'n/a' if overall is None else f'{overall:.2f}'}")
    return {"per_disorder_rsf": rsf, "overall_rsf": overall}

rsf_results = compute_rsf(self_agent_results, therapist_baseline_results)

EXPERIMENT 3 — RELATIVE SIMULATION FIDELITY
Paranoid personality disorder                 1.00
Schizoid personality disorder                 n/a
Schizotypal personality disorder              1.00
Antisocial personality disorder               1.00
Borderline personality disorder               1.00
Histrionic personality disorder               1.00
Narcissistic personality disorder             0.00
Avoidant personality disorder                 1.00
Dependent personality disorder                1.00
Obsessive-compulsive personality disorder     1.00

Overall RSF: 0.89


In [37]:
# ============================================================
# CHEAP RETEST — re-run only the report step on cases that already have a
# stored transcript, WITHOUT re-extracting or re-interviewing. Use this to
# check whether a REPORT_SYSTEM_PROMPT / REPORT_USER_TEMPLATE change actually
# moves a specific case, before spending money re-running the full batch.
#
# IMPORTANT: run the cell that defines REPORT_SYSTEM_PROMPT / REPORT_USER_TEMPLATE
# (the report-prompt cell) in THIS kernel session first -- generate_report_assessment
# reads those as globals, so an old prompt still in memory would silently be reused
# even though this notebook file has the new one on disk.
# ============================================================

def retest_cases_from_checkpoint(case_ids, checkpoint_path="experiment2_checkpoint.json",
                                  n_repeats=3, n_disorders=10, save=False):
    """Re-scores specific case_ids using their EXISTING transcript (from the
    checkpoint's 'history' field) run through generate_report_assessment n_repeats
    times, aggregated the same way evaluate_case_ensembled does. No interview turns,
    no extraction calls -- only report-generation calls (n_repeats per case_id).

    Prints an explicit before/after top-1 comparison for each case so you can see
    immediately whether it flipped.

    Set save=True to overwrite just these case_ids' entries in checkpoint_path in
    place -- every other case in the checkpoint is left untouched -- so your
    confusion-matrix / RSF cells reflect the update without re-running anything else.
    """
    with open(checkpoint_path) as f:
        ckpt = json.load(f)
    results_by_id = {str(r["case_id"]): r for r in ckpt["results"]}

    updated = []
    for cid in case_ids:
        cid = str(cid)
        old = results_by_id.get(cid)
        if old is None:
            print(f"case {cid!r} not found in {checkpoint_path}, skipping")
            continue
        if "history" not in old or not old["history"]:
            print(f"case {cid!r} has no stored transcript ('history'), skipping -- "
                  f"can't retest without re-running the interview for this one")
            continue

        transcript = "\n".join(f'{t["speaker"].capitalize()}: {t["text"]}' for t in old["history"])

        assessment_runs = []
        for i in range(n_repeats):
            parsed = generate_report_assessment(transcript, n_disorders=n_disorders)
            assessment_runs.append(parsed["assessment"])

        aggregated = aggregate_assessments(assessment_runs)
        scores = score_likelihoods(aggregated["assessment"], old["ground_truth"])

        new_result = {
            "case_id": cid, "ground_truth": old["ground_truth"],
            "predicted": aggregated["assessment"], "history": old["history"],
            **scores,
        }

        old_top1 = old.get("top1_prediction")
        new_top1 = new_result["top1_prediction"]
        flipped = "FLIPPED" if old_top1 != new_top1 else "unchanged"
        print(f"\n=== case {cid} (ground truth: {old['ground_truth']}) ===")
        print(f"  before: top1={old_top1!r} ({'correct' if old.get('top1_correct') else 'WRONG'})")
        print(f"  after:  top1={new_top1!r} ({'correct' if new_result['top1_correct'] else 'WRONG'}) "
              f"-- {flipped}")

        updated.append(new_result)

    if save and updated:
        by_id = {str(r["case_id"]): r for r in updated}
        ckpt["results"] = [by_id.get(str(r["case_id"]), r) for r in ckpt["results"]]
        with open(checkpoint_path, "w") as f:
            json.dump(ckpt, f, indent=2)
        print(f"\nSaved {len(updated)} updated case(s) back into {checkpoint_path}.")
    elif updated:
        print(f"\n(save=False -- nothing written to {checkpoint_path}. "
              f"Re-run with save=True once you're happy with the result.)")

    return updated


# Preview first (save=False costs the same either way -- this just controls
# whether the checkpoint file gets overwritten).
retested = retest_cases_from_checkpoint(["4", "6", "7"], save=True)



=== case 4 (ground truth: ['Antisocial personality disorder']) ===
  before: top1='Antisocial personality disorder' (correct)
  after:  top1='Antisocial personality disorder' (correct) -- unchanged

=== case 6 (ground truth: ['Histrionic personality disorder']) ===
  before: top1='Histrionic personality disorder' (correct)
  after:  top1='Histrionic personality disorder' (correct) -- unchanged

=== case 7 (ground truth: ['Narcissistic personality disorder']) ===
  before: top1='Paranoid personality disorder' (WRONG)
  after:  top1='Paranoid personality disorder' (WRONG) -- unchanged

Saved 3 updated case(s) back into experiment2_checkpoint.json.
